In [ ]:
# ===================== MEA/Neuralynx Spike Sorting — Jupyter UI =====================
# Tabs included: (1) Load & QC, (2) Sorting & Metrics, (3) Unit Curation
# - This UI calls the existing core functions from the Python module.
# - Keep parallelism in the core (ms4 workers, report workers, etc.). UI only passes values through.
# ========================================================================================
# ---- GPU pinning ----
import os
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import os, sys, importlib.util, threading, time, re, json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import ipywidgets as w
from IPython.display import display

# Optional file chooser
try:
    from ipyfilechooser import FileChooser
    HAS_FC = True
except Exception:
    HAS_FC = False

# Optional editable grid for per-channel toggles
try:
    from ipydatagrid import DataGrid
    HAS_GRID = True
except Exception:
    HAS_GRID = False

# ------------ BLAS oversubscription guard for 128-CPU nodes -------------
for _v in ("OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS","NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

# ------------------------- Core module dynamic loader ------------------------------
core_status = w.HTML("<em>Core not loaded</em>")
core_path   = w.Text(
    value="",
    placeholder="Full path to your core module (e.g., mea_pipeline_core.py)",
    description="Core .py:",
    layout=w.Layout(width="70%")
)
core_load_btn = w.Button(description="Load", icon="download", button_style="primary")

core = {}   # will hold bound functions and settings dataclass class

def _load_module(py_path: str):
    # Use a unique module name per path+time to avoid collisions when reloading different files
    safe_stem = Path(py_path).stem.replace("-", "_").replace(" ", "_")
    name = f"mea_core_{safe_stem}_{int(time.time())}"
 
    spec = importlib.util.spec_from_file_location(name, py_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not load: {py_path}")
 
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
 
    # <<< NEW: capture any prints during core import >>>
    sink = _ToWidget()
    with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
        spec.loader.exec_module(module)
    # >>> END NEW
 
    return module

def on_load(_):
    p = core_path.value.strip()
    if not p or not Path(p).exists():
        core_status.value = "<span style='color:#b00'> Path not found.</span>"
        return
    try:
        m = _load_module(p)
        # Bind the subset we need
        need = [
            "MEASettings",
            "ensure_spikeinterface_available",
            "load_mcs_h5",
            "load_neuralynx_folder",
            "discover_neuralynx_tetrode_groups",
            "load_neuralynx_tetrode_recording",
            "attach_probe_to_recording",
            "attach_tetrode_probe_to_recording",
            "ensure_probe_on",
            "preprocess_recording",
            "get_notch_effect_trace",
            "get_cmr_effect_trace",
            "get_channel_ids_compat",
            "compute_channel_rms_quietest",
            "subset_recording_channels",
            "run_mountainsort4",
            "run_kilosort4",          
            "run_spykingcircus2",     
            "extract_waveforms_and_pcs",
            "compute_unit_quality_metrics",
            "curate_units",
            "curate_quality_metrics_two_pass",
            "export_unit_spike_times",
            "make_unit_report_figures",
        ]
        missing = [n for n in need if not hasattr(m, n)]
        if missing:
            core_status.value = f"<span style='color:#b00'> Missing in core: {', '.join(missing)}</span>"
            return

        for n in need:
            core[n] = getattr(m, n)
        core["module"] = m  # keep module handle

        core_status.value = "<span style='color:#080'> Core loaded.</span>"
    except Exception as e:
        core_status.value = f"<span style='color:#b00'> Load failed: {e}</span>"

core_load_btn.on_click(on_load)

# ----------------------------- Log + Progress ---------------------------------------
log_out   = w.Output(layout=w.Layout(border="1px solid #ddd", height="280px", overflow="auto"))
progress  = w.IntProgress(value=0, min=0, max=100, description="Progress", bar_style='', layout=w.Layout(width="40%"))
def _log(msg: str):
    with log_out:
        ts = time.strftime("%H:%M:%S")
        print(f"[{ts}] {msg}")

def _mk_progress_cb():
    progress.value = 0
    def cb(done, total):
        progress.max = max(1, int(total))
        progress.value = min(max(0, int(done)), progress.max)
    return cb

def _finish_progress():
    progress.value = progress.max

import io, contextlib
 
class _ToWidget(io.TextIOBase):
    """Redirect prints into the UI log box without recursion."""
    def write(self, s):
        s = str(s)
        if not s:
            return 0
        # Split to keep line structure; write directly to the Output widget
        # (no print(), no _log() here — otherwise recursion)
        for line in s.splitlines():
            # keep raw tool output; add newline ourselves
            try:
                log_out.append_stdout(line + "\n")
            except Exception:
                # as a very last resort, swallow to avoid crashing the run
                pass
        return len(s)
 
    def flush(self):
        pass
    
# ----------------------------- Shared state -----------------------------------------
# Will hold a MEASettings instance and runtime objects (recording, preproc, sorting, waveforms)
state = dict(settings=None, recording=None, preproc=None, qc_df=None, include_ch=None, sorting=None, waveforms=None, metrics_df=None, input_mode="mcs_h5", input_path="", tetrode_groups=None, selected_tetrode="", active_run_dir=None)

def _ensure_settings():
    if state["settings"] is None:
        state["settings"] = core["MEASettings"]()

# ============================ TAB 1: Load & QC ======================================
# Inputs
input_mode_dd = w.Dropdown(
    options=[("MEA .h5", "mcs_h5"), ("Neuralynx tetrode folder (.ncs)", "neuralynx_tetrode")],
    value="mcs_h5",
    description="Input type",
    layout=w.Layout(width="360px")
)

if HAS_FC:
    h5_fc = FileChooser(str(Path.cwd()))
    h5_fc.title = "Select MEA .h5 or Neuralynx folder"
    geom_fc = FileChooser(str(Path.cwd()))
    geom_fc.title = "Geometry CSV (MEA mode only: hwid/channel,x_um,y_um)"
else:
    h5_tb   = w.Text(placeholder="Path to .h5 file OR Neuralynx folder", layout=w.Layout(width="70%"))
    geom_tb = w.Text(placeholder="Path to geometry CSV (MEA mode only)", layout=w.Layout(width="70%"))

tetrode_dd = w.Dropdown(options=[], description="Tetrode", layout=w.Layout(width="240px"))
scan_tetrodes_btn = w.Button(description="Scan tetrodes", icon="search")
tetrode_note = w.HTML("<em>Select a Neuralynx folder, scan tetrodes, then pick TT1/TT2/...</em>")
tetrode_members = w.HTML("<em>Members: </em>")
tetrode_box = w.VBox([w.HBox([tetrode_dd, scan_tetrodes_btn]), tetrode_note, tetrode_members])

outdir_tb = w.Text(value=str(Path.cwd()), description="Output folder:", layout=w.Layout(width="70%"))
pick_out  = w.Button(description="Use folder", icon="folder-open")
open_out  = w.Button(description="Open", icon="external-link-square")

# QC params
qc_frame = w.HBox()
qc_t0    = w.BoundedFloatText(value=0.0, min=0.0, step=0.1, description="Baseline start (s)", layout=w.Layout(width="230px"))
qc_t1    = w.BoundedFloatText(value=15.0, min=0.0, step=0.1, description="Baseline end (s)",   layout=w.Layout(width="230px"))
qc_win   = w.BoundedFloatText(value=15.0, min=0.01, max=60, step=0.05, description="Quiet length (s)",  layout=w.Layout(width="230px"))
hp_min   = w.FloatText(value=500.0, description="High-pass (Hz)",   layout=w.Layout(width="230px"))
hp_max   = w.FloatText(value=9000.0, description="Low-pass (Hz)",   layout=w.Layout(width="230px"))
use_cmr  = w.Checkbox(value=True, description="Common median reference (CMR)")
notch_filter = w.Checkbox(value=True, description="Apply notch filter")
notch_freq   = w.BoundedFloatText(value=60.0, min=10.0, max=200.0, step=1.0, description="Notch Freq (Hz)", layout=w.Layout(width="170px"))
run_qc   = w.Button(description="Compute QC", icon="check", button_style="success")

# Allow forcing fallback editor if ipydatagrid front-end isn't available
use_grid = w.Checkbox(value=False, description="Use editable grid (ipydatagrid)")

# QC table preview + select/deselect all
qc_tbl_btns = w.HBox()
qc_select_all   = w.Button(description="Select all")
qc_deselect_all = w.Button(description="Deselect all")
qc_apply_btn    = w.Button(description="Apply edits", icon="check", button_style="success")
qc_export_btn   = w.Button(description="Export QC to Excel", icon="file-excel-o")
qc_tbl_btns.children = [qc_select_all, qc_deselect_all, qc_apply_btn, qc_export_btn]

# Table will be shown via Pandas HTML; we keep the DataFrame in state["qc_df"]
qc_html = w.HTML("<em>QC table appears here after computation.</em>")
qc_table_box = w.VBox([qc_html])   # container that will hold either HTML or an editable grid
qc_grid = None                     # will hold DataGrid if available

# Fallback editor used when the grid is unavailable/off
qc_exclude_list = w.SelectMultiple(
    options=[],                 # filled in at runtime
    description="Exclude:",
    layout=w.Layout(width="320px", height="240px")
)

# --- Flagged-channel quick preview ---
qc_flagged_dropdown = w.Dropdown(options=[], description="Flagged:", layout=w.Layout(width="240px"))
qc_plot_btn = w.Button(description="Plot 60 s", icon="line-chart")
qc_plot_out = w.Output(layout=w.Layout(border="1px solid #ddd", height="240px", overflow="auto"))
qc_allch_dropdown = w.Dropdown(options=[], description="Channel:", layout=w.Layout(width="240px"))
qc_plot_notch_btn = w.Button(description="Plot notch", icon="bar-chart")
qc_plot_cmr_btn = w.Button(description="Plot CMR", icon="area-chart")
qc_plot_notch_out = w.Output(layout=w.Layout(border="1px solid #ddd", height="240px", overflow="auto"))
qc_plot_cmr_out = w.Output(layout=w.Layout(border="1px solid #ddd", height="240px", overflow="auto"))

def _update_flagged_controls():
    if state["qc_df"] is None:
        qc_flagged_dropdown.options = []
        return
    flagged = state["qc_df"].loc[state["qc_df"]["auto_flag"], "channel_id"].tolist()
    qc_flagged_dropdown.options = flagged or []

def _get_input_path():
    val = (h5_fc.selected if HAS_FC else h5_tb.value)
    return (val or "").strip()

def _get_h5():
    return _get_input_path()

def _get_geom():
    val = (geom_fc.selected if HAS_FC else geom_tb.value)
    return (val or "").strip()

def _get_mode():
    return input_mode_dd.value

def _safe_label(text: str) -> str:
    return re.sub(r'[^A-Za-z0-9._-]+', '_', str(text)).strip('_') or 'session'

def _get_active_label():
    mode = _get_mode()
    in_path = _get_input_path()
    if mode == 'neuralynx_tetrode':
        base = Path(in_path).name if in_path else 'neuralynx'
        tet = tetrode_dd.value or state.get('selected_tetrode') or 'TT'
        return _safe_label(f"{base}_{tet}")
    return _safe_label(Path(in_path).stem if in_path else 'session')

def _get_active_run_dir(base_outdir: str = None):
    base = Path(base_outdir or (outdir_tb.value.strip() or str(Path.cwd())))
    label = _get_active_label()
    if _get_mode() == 'neuralynx_tetrode':
        run_dir = base if base.name == label else (base / label)
    else:
        run_dir = base
    run_dir.mkdir(parents=True, exist_ok=True)
    state['active_run_dir'] = str(run_dir)
    return str(run_dir)

def _refresh_mode_visibility():
    is_nlx = (_get_mode() == 'neuralynx_tetrode')
    tetrode_box.layout.display = None if is_nlx else 'none'
    geom_row.layout.display = 'none' if is_nlx else None
    if not is_nlx:
        tetrode_members.value = "<em>Members: </em>"
    if HAS_FC:
        h5_fc.title = 'Select Neuralynx folder' if is_nlx else 'Select MEA .h5'
        try:
            h5_fc.show_only_dirs = bool(is_nlx)
        except Exception:
            pass
    else:
        h5_tb.placeholder = 'Path to Neuralynx folder' if is_nlx else 'Path to .h5 file'

def _refresh_tetrode_members(*args):
    groups = state.get('tetrode_groups') or {}
    tet = tetrode_dd.value
    members = groups.get(tet, []) if tet else []
    if members:
        tetrode_members.value = "<b>Members:</b> " + ", ".join(map(str, members))
    else:
        tetrode_members.value = "<em>Members: </em>"

def on_scan_tetrodes(_=None):
    if not core:
        _log('Load the core module first.')
        return
    folder = _get_input_path()
    if not folder or not Path(folder).exists():
        _log('Provide a valid Neuralynx folder path first.')
        return
    try:
        groups = core['discover_neuralynx_tetrode_groups'](folder)
        state['tetrode_groups'] = groups
        options = list(groups.keys())
        tetrode_dd.options = options
        if options and (tetrode_dd.value not in options):
            tetrode_dd.value = options[0]
        _refresh_tetrode_members()
        if options:
            _log(f"Discovered tetrodes: {', '.join(options)}")
        else:
            _log('No tetrode groups found in that folder.')
    except Exception as e:
        tetrode_dd.options = []
        state['tetrode_groups'] = None
        tetrode_members.value = "<em>Members: </em>"
        _log(f'Failed to scan tetrodes: {e}')

def _refresh_qc_table():
    global qc_grid, HAS_GRID 
 
    if state["qc_df"] is None or state["qc_df"].empty:
        qc_table_box.children = [w.HTML("<em>No QC results yet.</em>")]
        return
 
    show_cols = ["channel_id", "rms", "rms_thr", "auto_flag", "threshold_3x", "include", "quiet_start_s", "quiet_end_s"]
    for c in show_cols:
        if c not in state["qc_df"].columns:
            state["qc_df"][c] = np.nan if c not in ("include","auto_flag") else (False if c=="auto_flag" else True)
    df = state["qc_df"][show_cols].copy()
 
    # Try rich grid first (only if toggle is ON and front-end is available)
    if HAS_GRID and use_grid.value:
        df["include"] = df["include"].astype(bool)
        try:
            if qc_grid is None:
                qc_grid = DataGrid(
                    df,
                    editable=True,
                    selection_mode="row",
                    base_column_options={"editable": False},           
                    column_definitions={"include": {"editable": True}} 
                )
                qc_grid.layout = w.Layout(height="380px", border="1px solid #ddd")
            else:
                qc_grid.dataframe = df
            qc_table_box.children = [qc_grid]
            return
        except Exception:
            # Front-end couldn't load ipydatagrid -> use fallback and don't try again
            HAS_GRID = False
 
    # ---------- Fallback: read-only table + editable "Exclude" list ----------
    df_fmt = df.copy()
    for c, fmt in [("rms", "{:.4f}"), ("threshold_3x", "{:.4f}"),
                   ("quiet_start_s", "{:.3f}"), ("quiet_end_s", "{:.3f}")]:
        df_fmt[c] = df_fmt[c].astype(float).map(lambda x: fmt.format(x))
    qc_html.value = df_fmt.to_html(index=False)
 
    # Populate the Exclude list from current 'include' flags
    all_ch   = df["channel_id"].tolist()
    excluded = df.loc[~df["include"], "channel_id"].tolist()
    qc_exclude_list.options = all_ch
    qc_exclude_list.value   = tuple(excluded)
 
    # Show table alongside the Exclude editor
    qc_table_box.children = [
        w.HBox([
            qc_html,
            w.VBox([w.HTML("<b>Per-channel selection</b>"), qc_exclude_list])
        ])
    ]
    
def _sync_qc_from_ui():
    """Sync UI edits back into state['qc_df'] (grid or fallback)."""
    if state["qc_df"] is None:
        return
 
    # Grid path
    if HAS_GRID and use_grid.value and qc_grid is not None:
        try:
            df_ui = qc_grid.dataframe.copy()
            df_ui["include"] = df_ui["include"].astype(bool)
            inc_map = dict(zip(df_ui["channel_id"], df_ui["include"]))
            state["qc_df"]["include"] = (
                state["qc_df"]["channel_id"].map(inc_map).fillna(state["qc_df"]["include"]).astype(bool)
            )
        except Exception:
            pass
    else:
        # Fallback: 'Exclude' list -> include = channel_id not in that list
        excluded = set(qc_exclude_list.value)
        state["qc_df"]["include"] = ~state["qc_df"]["channel_id"].isin(excluded)
 
    state["include_ch"] = list(state["qc_df"].loc[state["qc_df"]["include"], "channel_id"].values)
    
def on_pick_out(_):
    # Just trust what's in outdir_tb; ensure it exists
    p = outdir_tb.value.strip() or str(Path.cwd())
    Path(p).mkdir(parents=True, exist_ok=True)
    _ensure_settings()
    state["settings"].outdir = p
    _log(f"Output folder set: {p}")

def on_open_out(_):
    p = outdir_tb.value.strip() or str(Path.cwd())
    if not Path(p).exists():
        _log("Output folder does not exist.")
        return
    try:
        if sys.platform.startswith('darwin'):
            os.system(f'open "{p}"')
        elif os.name == 'nt':
            os.startfile(p)  # noqa
        else:
            os.system(f'xdg-open "{p}"')
    except Exception:
        pass

def _bg(fn, *args, **kwargs):
    def _wrap():
        try:
            fn(*args, **kwargs)
        finally:
            pass
    threading.Thread(target=_wrap, daemon=True).start()

def on_run_qc(_):
    log_out.clear_output()
    progress.value = 0
    if not core:
        _log("Load the core module first.")
        return
    sink = _ToWidget()

    input_path = _get_input_path()
    mode = _get_mode()
    if not input_path or not Path(input_path).exists():
        _log("Provide a valid input path.")
        return

    base_outdir = outdir_tb.value.strip() or str(Path.cwd())
    Path(base_outdir).mkdir(parents=True, exist_ok=True)
    _ensure_settings()
    S = state["settings"]
    S.input_mode = mode
    S.input_path = input_path
    S.input_h5 = input_path if mode == 'mcs_h5' else ''
    S.input_folder = input_path if mode == 'neuralynx_tetrode' else ''
    S.outdir = _get_active_run_dir(base_outdir)
    S.hp_min_hz = float(hp_min.value)
    S.hp_max_hz = float(hp_max.value)
    S.use_cmr        = bool(use_cmr.value)
    S.use_notch_60hz = bool(notch_filter.value)
    S.hp_notch_hz    = float(notch_freq.value)
    S.apply_notch    = S.use_notch_60hz
    S.notch_freq_hz  = S.hp_notch_hz

    scan_start = float(qc_t0.value)
    scan_end   = float(qc_t1.value)
    win_s      = float(qc_win.value)

    if scan_end <= scan_start:
        _log("Baseline end must be > start. Adjusting end = start + 1.0 s.")
        scan_end = scan_start + 1.0
        qc_t1.value = scan_end
    if win_s <= 0:
        _log("Quiet length must be > 0. Setting to 1.0 s.")
        win_s = 1.0
        qc_win.value = win_s
    if (scan_end - scan_start) < win_s:
        _log("Scan window shorter than quiet length. Expanding end to fit.")
        scan_end = scan_start + win_s
        qc_t1.value = scan_end

    _log(f"Loading recording [{mode}]: {input_path}")
    try:
        with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
            core["ensure_spikeinterface_available"]()
            if mode == 'neuralynx_tetrode':
                groups = state.get('tetrode_groups')
                if not groups:
                    groups = core['discover_neuralynx_tetrode_groups'](input_path)
                    state['tetrode_groups'] = groups
                    tetrode_dd.options = list(groups.keys())
                    if tetrode_dd.options and (tetrode_dd.value not in tetrode_dd.options):
                        tetrode_dd.value = tetrode_dd.options[0]
                tetrode_name = tetrode_dd.value
                if not tetrode_name:
                    raise RuntimeError('No tetrode selected. Click Scan tetrodes and choose one.')
                state['selected_tetrode'] = tetrode_name
                S.selected_tetrode = tetrode_name
                S.outdir = _get_active_run_dir(base_outdir)
                stage_root = str(Path(S.outdir) / '_neuralynx_staging')
                rec = core['load_neuralynx_tetrode_recording'](
                    input_path,
                    tetrode_name=tetrode_name,
                    staged_root=stage_root,
                    spacing_um=float(getattr(S, 'tetrode_spacing_um', 25.0)),
                    layout=str(getattr(S, 'tetrode_layout', 'square')),
                )
            else:
                rec = core["load_mcs_h5"](input_path)
        state["recording"] = rec
        state['input_mode'] = mode
        state['input_path'] = input_path
        ids = list(rec.get_channel_ids()) if hasattr(rec, 'get_channel_ids') else []
        qc_allch_dropdown.options = ids or []
    except Exception as e:
        _log(f"Failed to load input: {e}")
        return

    # Geometry is only user-supplied in MEA mode; tetrode mode attaches compact geometry automatically
    if mode == 'mcs_h5':
        geom = _get_geom()
        if geom:
            try:
                _log(f"Assigning geometry from CSV: {geom}")
                gdf = pd.read_csv(geom)
                cols = {c.lower() for c in gdf.columns}
                if not (("hwid" in cols or "channel" in cols) and "x_um" in cols and "y_um" in cols):
                    raise ValueError("CSV must have either 'hwid' or 'channel', plus 'x_um','y_um'.")
                gdf = gdf.rename(columns={c: c.lower() for c in gdf.columns})
                if "hwid" in gdf.columns:
                    gdf["hwid_zero"] = gdf["hwid"].astype(int) - 1
                else:
                    gdf["hwid_zero"] = gdf["channel"].astype(int)
                gdf["hwid_str"] = gdf["hwid_zero"].apply(lambda n: f"Ch{int(n)}")

                ch_ids = state["recording"].get_channel_ids()
                geom_map_str = dict(zip(gdf["hwid_str"], zip(gdf["x_um"], gdf["y_um"])))
                geom_map_num = dict(zip(gdf["hwid_zero"].astype(str), zip(gdf["x_um"], gdf["y_um"])))
                locs, matched, key_used = [], [], []
                hits = 0
                for ch in ch_ids:
                    key = str(ch)
                    if key in geom_map_str:
                        xy = geom_map_str[key]; hits += 1; matched.append(True); key_used.append(key)
                    elif key in geom_map_num:
                        xy = geom_map_num[key]; hits += 1; matched.append(True); key_used.append(key)
                    else:
                        if isinstance(ch, str) and ch.startswith("Ch"):
                            alt = ch[2:]; xy = geom_map_num.get(alt, (np.nan, np.nan))
                            matched.append(alt in geom_map_num); key_used.append(f"(alt){alt}" if alt in geom_map_num else "")
                            if alt in geom_map_num: hits += 1
                        else:
                            alt = f"Ch{ch}"; xy = geom_map_str.get(alt, (np.nan, np.nan))
                            matched.append(alt in geom_map_str); key_used.append(f"(alt){alt}" if alt in geom_map_str else "")
                            if alt in geom_map_str: hits += 1
                    locs.append(xy)
                locs = np.asarray(locs, dtype=np.float32)
                with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
                    rec2 = core["attach_probe_to_recording"](state["recording"], locs, ch_ids, electrode_diameter_um=30.0, ied_um=200.0)
                state["recording"] = rec2
                check_df = pd.DataFrame({"channel_id": ch_ids, "matched": matched, "key_used": key_used, "x_um": locs[:,0], "y_um": locs[:,1]})
                check_path = str(Path(S.outdir)/"geometry_assignment_check.csv")
                check_df.to_csv(check_path, index=False)
                _log(f"Geometry assigned ({hits}/{len(ch_ids)}). Check: {check_path}")
            except Exception as e:
                _log(f"Geometry assignment issue: {e}")
    else:
        _log(f"Using auto-generated compact tetrode geometry for {state.get('selected_tetrode') or tetrode_dd.value}.")

    _log("Preprocessing (bandpass/CMR)…")
    try:
        with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
            pre = core["preprocess_recording"](state["recording"], S)
            pre = core["ensure_probe_on"](pre, state["recording"])
        state["preproc"] = pre
    except Exception as e:
        _log(f"Preprocess error: {e}")
        return

    _log(f"QC: quietest {win_s:.3g}s in [{scan_start:.3g}, {scan_end:.3g}]…")
    try:
        with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
            df = core["compute_channel_rms_quietest"](state["preproc"], scan_start_s=scan_start, scan_end_s=scan_end, window_s=win_s)
        state["qc_df"] = df.copy()

        r = pd.to_numeric(state["qc_df"]["rms"], errors="coerce")
        mu = float(np.nanmean(r))
        sd = float(np.nanstd(r, ddof=1)) if np.isfinite(np.nanstd(r, ddof=1)) else float(np.nanstd(r))
        thr = mu + 2.0 * sd
        state["qc_df"]["rms_thr"]   = thr
        state["qc_df"]["auto_flag"] = state["qc_df"]["rms"] > thr

        auto_exclude = not (mode == 'neuralynx_tetrode' and len(state['qc_df']) <= 4)
        if auto_exclude:
            state["qc_df"]["include"] = state["qc_df"]["include"] & (~state["qc_df"]["auto_flag"])
        else:
            _log('Neuralynx tetrode mode detected: flagged channels are highlighted, but not auto-excluded.')

        state["include_ch"] = list(state["qc_df"].loc[state["qc_df"]["include"], "channel_id"].values)
        _refresh_qc_table()
        _update_flagged_controls()
        n_flag = int(state["qc_df"]["auto_flag"].sum())
        _log(f"QC done. Channels: {len(df)}; Included: {len(state['include_ch'])} | Auto-flagged (> {thr:.4f}): {n_flag}")
        _log("QC table mode: " + ("grid" if (HAS_GRID and use_grid.value) else "fallback"))
    except Exception as e:
        _log(f"QC error: {e}")
pick_out.on_click(on_pick_out)
open_out.on_click(on_open_out)
run_qc.on_click(on_run_qc)
scan_tetrodes_btn.on_click(on_scan_tetrodes)
tetrode_dd.observe(_refresh_tetrode_members, 'value')
def _on_input_mode_change(change=None):
    _refresh_mode_visibility()
    try:
        _apply_all_sorter_defaults_for_mode(force=True)
    except NameError:
        pass
    try:
        _update_sorter_explanation()
    except NameError:
        pass

input_mode_dd.observe(_on_input_mode_change, names="value")

def _set_all(val: bool):
    if state["qc_df"] is None:
        return
    state["qc_df"]["include"] = bool(val)
    state["include_ch"] = list(state["qc_df"].loc[state["qc_df"]["include"], "channel_id"].values)
    _refresh_qc_table()

qc_select_all.on_click(lambda _: _set_all(True))
qc_deselect_all.on_click(lambda _: _set_all(False))

def on_qc_apply(_):
    _sync_qc_from_ui()
    _refresh_qc_table()
    _excluded = int((~state["qc_df"]["include"]).sum())
    _log(f"Applied QC edits. Included: {len(state['include_ch'])} | Excluded: {_excluded}")

qc_apply_btn.on_click(on_qc_apply)

def on_qc_export(_):
    if state["qc_df"] is None:
        _log("Run QC first.")
        return
    _sync_qc_from_ui()  
    outdir = outdir_tb.value.strip() or str(Path.cwd())
    out = str(Path(_get_active_run_dir(outdir))/"qc_channels.xlsx")
    state["qc_df"].to_excel(out, index=False)
    _log(f"QC saved: {out}")

qc_export_btn.on_click(on_qc_export)

def on_qc_plot(_):
    ch = qc_flagged_dropdown.value
    if ch is None:
        _log("No flagged channel selected.")
        return
    if state["preproc"] is None:
        _log("Preprocessed recording not available.")
        return
 
    try:
        rec = state["preproc"]
        ch_ids = list(rec.get_channel_ids())
 
        # --- Normalize selected channel to match recording IDs ---
        candidates = [ch]
        try:
            candidates.append(int(ch))
        except Exception:
            pass
        candidates.append(str(ch))
        try:
            candidates.append(f"Ch{int(ch)}")
        except Exception:
            candidates.append(f"Ch{ch}")
        ch_key = None
        for c in candidates:
            if c in ch_ids:
                ch_key = c
                break
        if ch_key is None:
            ch_str = str(ch)
            for cid in ch_ids:
                if str(cid).lstrip("Ch").lstrip("ch") == ch_str.lstrip("Ch").lstrip("ch"):
                    ch_key = cid
                    break
        if ch_key is None:
            _log(f"Channel {ch} not found in recording IDs (first few: {ch_ids[:5]} …)")
            return
 
        # --- Grab first 60 s, lightly downsample for plotting ---
        fs = float(rec.get_sampling_frequency())
        n_frames = int(min(rec.get_num_frames(segment_index=0), fs * 60.0))
        tr = rec.get_traces(start_frame=0, end_frame=n_frames, channel_ids=[ch_key], segment_index=0).ravel()
        step = max(1, len(tr) // 20000)  # at most ~20k points to draw
        t = (np.arange(0, len(tr), step, dtype=float) / fs)
        y = tr[::step]
 
        # --- Render to PNG so it displays regardless of matplotlib backend ---
        import matplotlib.pyplot as plt
        from io import BytesIO
        from IPython.display import Image, display
 
        with qc_plot_out:
            qc_plot_out.clear_output(wait=True)
            fig = plt.figure(figsize=(7, 2.8))
            ax = fig.add_subplot(111)
            ax.plot(t, y)
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("µV")
            ax.set_title(f"Ch {ch_key} — first 60 s")
            fig.tight_layout()
 
            buf = BytesIO()
            fig.savefig(buf, format="png", dpi=130)
            plt.close(fig)
            buf.seek(0)
            display(Image(data=buf.getvalue()))
    except Exception as e:
        _log(f"Plot error: {e}")
        
qc_plot_btn.on_click(on_qc_plot)

def on_plot_notch(_):
    ch = qc_allch_dropdown.value
    if ch is None:
        _log("No channel selected.")
        return
    if state["recording"] is None:
        _log("Recording not loaded.")
        return
    if "get_notch_effect_trace" not in core:
        _log("Loaded core does not expose get_notch_effect_trace.")
        return
    try:
        t, tr_pre, tr_post, ch_key = core["get_notch_effect_trace"](
            state["recording"], state["settings"], ch, seconds=60
        )
        step = max(1, len(tr_pre) // 5000)
        t = t[::step]
        y_pre = tr_pre[::step]
        y_post = tr_post[::step]

        import matplotlib.pyplot as plt
        from io import BytesIO
        from IPython.display import Image, display

        with qc_plot_notch_out:
            qc_plot_notch_out.clear_output(wait=True)
            fig = plt.figure(figsize=(7, 2.8))
            ax = fig.add_subplot(111)
            ax.plot(t, y_pre, label="Before notch (post-bandpass)", alpha=0.6)
            ax.plot(t, y_post, label="After notch", alpha=0.6)
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("µV")
            ax.set_title(f"Ch {ch_key} — pre/post notch")
            ax.legend()
            fig.tight_layout()

            buf = BytesIO()
            fig.savefig(buf, format="png", dpi=130)
            plt.close(fig)
            buf.seek(0)
            display(Image(data=buf.getvalue()))
    except Exception as e:
        _log(f"Plot error: {e}")

qc_plot_notch_btn.on_click(on_plot_notch)

def on_plot_cmr(_):
    ch = qc_allch_dropdown.value
    if ch is None:
        _log("No channel selected.")
        return
    if state["recording"] is None:
        _log("Recording not loaded.")
        return
    if "get_cmr_effect_trace" not in core:
        _log("Loaded core does not expose get_cmr_effect_trace.")
        return
    try:
        t, tr_pre, tr_post, ch_key = core["get_cmr_effect_trace"](
            state["recording"], state["settings"], ch, seconds=60
        )
        step = max(1, len(tr_pre) // 5000)
        t = t[::step]
        y_pre = tr_pre[::step]
        y_post = tr_post[::step]

        import matplotlib.pyplot as plt
        from io import BytesIO
        from IPython.display import Image, display

        with qc_plot_cmr_out:
            qc_plot_cmr_out.clear_output(wait=True)
            fig = plt.figure(figsize=(7, 2.8))
            ax = fig.add_subplot(111)
            ax.plot(t, y_pre, label="Before CMR (post-notch)", alpha=0.6)
            ax.plot(t, y_post, label="After CMR", alpha=0.6)
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("µV")
            ax.set_title(f"Ch {ch_key} — pre/post CMR")
            ax.legend()
            fig.tight_layout()

            buf = BytesIO()
            fig.savefig(buf, format="png", dpi=130)
            plt.close(fig)
            buf.seek(0)
            display(Image(data=buf.getvalue()))
    except Exception as e:
        _log(f"Plot error: {e}")

qc_plot_cmr_btn.on_click(on_plot_cmr)

# =========================== TAB 2: Sorting & Metrics ===============================
# ---- Sorter select ----
sorter_dd = w.Dropdown(
    options=[("MountainSort4","mountainsort4"),
             ("Kilosort4","kilosort4"),
             ("SpykingCircus2","spykingcircus2")],
    value="mountainsort4",
    description="Sorter",
    layout=w.Layout(width="260px")
)

# --- MountainSort4 parameters (with presets) ---
# Base widgets
det_thr   = w.BoundedFloatText(value=3.0, min=1.0, step=0.1, description="detect_threshold", layout=w.Layout(width="220px"))
det_sign  = w.IntText(value=0, description="detect_sign (-1/0/+1)", layout=w.Layout(width="220px"))
adj_rad   = w.BoundedFloatText(value=200.0, min=-1.0, max=100000.0, step=5.0, description="adjacency_radius (µm)", layout=w.Layout(width="220px"))
clip_size = w.BoundedIntText(value=128, min=20, max=512, step=2, description="clip_size (samples)", layout=w.Layout(width="220px"))
ms4_workers = w.BoundedIntText(
    value=12, min=1, max=os.cpu_count() or 256,
    description="ms4 workers", layout=w.Layout(width="220px")
)
det_int = w.BoundedIntText(  # detect_interval (duplicate-suppression window, samples)
    value=10, min=1, max=400,
    description="detect_interval (samples)", layout=w.Layout(width="260px")
)

# Preset dropdown that autofills the widgets
ms4_preset_dd = w.Dropdown(
    options=["Local (200 µm)", "Wide (400 µm)", "Global-ish (-1)"],
    value="Local (200 µm)",
    description="MS4 preset",
    layout=w.Layout(width="240px")
)

MS4_MEA_DEFAULTS = {
    "detect_threshold": 4.0,
    "detect_sign": 0,
    "clip_size": 128,
    "detect_interval": 10,
    "ms4_workers": 12,
}

MS4_NLX_DEFAULTS = {
    "detect_threshold": 4.0,
    "detect_sign": 0,
    "adjacency_radius": 80.0,
    "clip_size": 128,
    "detect_interval": 16,
    "ms4_workers": 12,
}

def _set_ms4_widgets(*, detect_threshold=None, detect_sign_val=None, adjacency_radius=None,
                     clip_size_val=None, detect_interval=None, workers=None):
    if detect_threshold is not None:
        det_thr.value = float(detect_threshold)
    if detect_sign_val is not None:
        det_sign.value = int(detect_sign_val)
    if adjacency_radius is not None:
        adj_rad.value = float(adjacency_radius)
    if clip_size_val is not None:
        clip_size.value = int(clip_size_val)
    if detect_interval is not None:
        det_int.value = int(detect_interval)
    if workers is not None:
        ms4_workers.value = int(workers)

def _fill_ms4_from_preset(change=None):
    name = ms4_preset_dd.value
    _set_ms4_widgets(
        detect_threshold=MS4_MEA_DEFAULTS["detect_threshold"],
        detect_sign_val=MS4_MEA_DEFAULTS["detect_sign"],
        clip_size_val=MS4_MEA_DEFAULTS["clip_size"],
        detect_interval=MS4_MEA_DEFAULTS["detect_interval"],
        workers=MS4_MEA_DEFAULTS["ms4_workers"],
    )

    if "Local" in name:
        adj_rad.value = 200.0
    elif "Wide" in name:
        adj_rad.value = 400.0
    else:  # Global-ish
        adj_rad.value = -1.0    # SI/MS4 "global" adjacency

def _apply_ms4_defaults_for_mode(force=False):
    is_nlx = (_get_mode() == "neuralynx_tetrode")
    ms4_preset_dd.layout.display = "none" if is_nlx else None

    if is_nlx:
        _set_ms4_widgets(
            detect_threshold=MS4_NLX_DEFAULTS["detect_threshold"],
            detect_sign_val=MS4_NLX_DEFAULTS["detect_sign"],
            adjacency_radius=MS4_NLX_DEFAULTS["adjacency_radius"],
            clip_size_val=MS4_NLX_DEFAULTS["clip_size"],
            detect_interval=MS4_NLX_DEFAULTS["detect_interval"],
            workers=MS4_NLX_DEFAULTS["ms4_workers"],
        )
    elif force:
        _fill_ms4_from_preset()

# initialize + wire up
_fill_ms4_from_preset()
ms4_preset_dd.observe(_fill_ms4_from_preset, "value")

# Wrap MS4 params in a box so we can toggle visibility
ms4_params_box = w.VBox([
    w.HTML("<b>MountainSort4 parameters</b>"),
    ms4_preset_dd,
    w.HBox([det_thr, det_sign, adj_rad, clip_size, det_int, ms4_workers]),
])

# --- Kilosort4 parameters panel (with full presets) ---
ks4_info = w.HTML(
    "<b>Kilosort4 parameters</b><br>"
    "Choose a preset or edit the JSON. These map to SpikeInterface Kilosort4 options."
)

# Presets tuned for MEA 20 kHz, 200 µm pitch, 30 µm dia, both signs
ks4_presets = {
    "Local": """{
  "fs": 20000,
  "highpass_cutoff": 300,
  "Th_universal": 9,
  "Th_learned": 8,

  "nt": 61,
  "nblocks": 1,
  "do_correction": false,

  "do_CAR": true,
  "torch_device": "auto",

  "dmin": 200,
  "dminx": 200,
  "max_channel_distance": 300,
  "whitening_range": 32,
  "nearest_chans": 10,
  "nearest_templates": 100,

  "duplicate_spike_ms": 0.25,
  "templates_from_data": true,

  "cluster_neighbors": 10,
  "cluster_downsampling": 1,
  "max_cluster_subset": 25000,

  "skip_kilosort_preprocessing": false,
  "keep_good_only": false,
  "use_binary_file": true,
  "delete_recording_dat": true,

  "clear_cache": false,
  "progress_bar": true
}""",
    "Wide": """{
  "fs": 20000,
  "highpass_cutoff": 300,
  "Th_universal": 9,
  "Th_learned": 8,

  "nt": 61,
  "nblocks": 1,
  "do_correction": false,

  "do_CAR": true,
  "torch_device": "auto",

  "dmin": 400,
  "dminx": 400,
  "max_channel_distance": 600,
  "whitening_range": 48,
  "nearest_chans": 16,
  "nearest_templates": 150,

  "duplicate_spike_ms": 0.25,
  "templates_from_data": true,

  "cluster_neighbors": 12,
  "cluster_downsampling": 1,
  "max_cluster_subset": 30000,

  "skip_kilosort_preprocessing": false,
  "keep_good_only": false,
  "use_binary_file": true,
  "delete_recording_dat": true,

  "clear_cache": false,
  "progress_bar": true
}""",
    "Global-ish": """{
  "fs": 20000,
  "highpass_cutoff": 300,
  "Th_universal": 9,
  "Th_learned": 8,

  "nt": 61,
  "nblocks": 1,
  "do_correction": false,

  "do_CAR": true,
  "torch_device": "auto",

  "dmin": 100000,
  "dminx": 100000,
  "max_channel_distance": 100000,
  "whitening_range": 64,
  "nearest_chans": 32,
  "nearest_templates": 200,

  "duplicate_spike_ms": 0.25,
  "templates_from_data": true,

  "cluster_neighbors": 15,
  "cluster_downsampling": 1,
  "max_cluster_subset": 40000,

  "skip_kilosort_preprocessing": false,
  "keep_good_only": false,
  "use_binary_file": true,
  "delete_recording_dat": true,

  "clear_cache": false,
  "progress_bar": true
}"""
}

# Presets tuned for Nlx 32kHz, tetrode, both signs
KS4_NLX_DEFAULT_JSON = """{
  "fs": 32000,
  "highpass_cutoff": 500,
  "Th_universal": 9,
  "Th_learned": 8,

  "nt": 129,
  "nblocks": 1,
  "do_correction": false,

  "do_CAR": true,
  "torch_device": "auto",

  "dmin": 25,
  "dminx": 25,
  "max_channel_distance": 80,
  "whitening_range": 4,
  "nearest_chans": 4,
  "nearest_templates": 50,

  "duplicate_spike_ms": 0.5,
  "templates_from_data": true,

  "cluster_neighbors": 4,
  "cluster_downsampling": 1,
  "max_cluster_subset": 20000,

  "skip_kilosort_preprocessing": false,
  "keep_good_only": false,
  "use_binary_file": true,
  "delete_recording_dat": true,

  "clear_cache": false,
  "progress_bar": true
}"""

ks4_preset_dd = w.Dropdown(
    options=["Local", "Wide", "Global-ish"],
    value="Local",
    description="KS4 preset",
    layout=w.Layout(width="240px")
)

ks4_params_json = w.Textarea(
    value="",
    placeholder="Edit KS4 preset JSON here…",
    description="Advanced JSON",
    layout=w.Layout(width="540px", height="160px")
)

def _fill_ks4_from_preset(change=None):
    if _get_mode() == "neuralynx_tetrode":
        ks4_params_json.value = KS4_NLX_DEFAULT_JSON
    else:
        ks4_params_json.value = ks4_presets[ks4_preset_dd.value]

def _apply_ks4_defaults_for_mode(force=False):
    is_nlx = (_get_mode() == "neuralynx_tetrode")
    ks4_preset_dd.layout.display = "none" if is_nlx else None
    if is_nlx:
        if force or not ks4_params_json.value.strip():
            ks4_params_json.value = KS4_NLX_DEFAULT_JSON
    elif force:
        ks4_params_json.value = ks4_presets[ks4_preset_dd.value]

_fill_ks4_from_preset()
ks4_preset_dd.observe(_fill_ks4_from_preset, "value")

ks4_params_box = w.VBox([ks4_info, ks4_preset_dd, ks4_params_json])
ks4_params_box.layout.display = "none"  # shown only when KS4 selected

# --- SpykingCircus2 parameters panel (full dictionaries + presets) ---
sc2_info = w.HTML(
    "<b>SpykingCircus2 parameters</b><br>"
    "Choose a preset or edit the JSON. Includes SC2 dicts: general, sparsity, filtering, "
    "whitening, detection, selection, clustering, matching, merging, motion_correction, "
    "and runtime controls."
)

# Presets tuned for MEA 20 kHz, 200 µm pitch, 30 µm dia, both signs
sc2_presets = {
    "Local": """{
  "apply_preprocessing": true,
  "apply_motion_correction": false,
  "matched_filtering": true,
  "templates_from_svd": true,
  "multi_units_only": false,
  "job_kwargs": { "n_jobs": 12 },
  "seed": 42,
  "deterministic_peaks_detection": true,

  "general": { "ms_before": 1.5, "ms_after": 2.5, "radius_um": 200 },
  "sparsity": { "method": "radius", "radius_um": 200 },

  "filtering": { "type": "butterworth", "high_pass": 300, "order": 3 },
  "whitening": { "mode": "local", "neighbors": 32 },

  "detection": { "peak_sign": "both", "detect_threshold": 3.0, "min_peak_distance_ms": 1.2 },
  "selection": { "method": "uniform", "n_peaks": 20000 },
  "clustering": { "method": "graph_clustering", "k_nearest": 10 },
  "matching": { "method": "circus-omp-svd" },

  "merging": { "auto_merge_units": true, "acg_threshold": 0.2, "ccg_threshold": 0.25 },

  "motion_correction": { "enabled": false },

  "cache_preprocessing": { "mode": "memory", "memory_limit": "2G", "delete_cache": true },
  "chunk_preprocessing": { "n_jobs": 12, "total_memory": "4G" }
}""",
    "Wide": """{
  "apply_preprocessing": true,
  "apply_motion_correction": false,
  "matched_filtering": true,
  "templates_from_svd": true,
  "multi_units_only": false,
  "job_kwargs": { "n_jobs": 12 },
  "seed": 42,
  "deterministic_peaks_detection": true,

  "general": { "ms_before": 1.5, "ms_after": 2.5, "radius_um": 400 },
  "sparsity": { "method": "radius", "radius_um": 400 },

  "filtering": { "type": "butterworth", "high_pass": 300, "order": 3 },
  "whitening": { "mode": "local", "neighbors": 48 },

  "detection": { "peak_sign": "both", "detect_threshold": 3.0, "min_peak_distance_ms": 1.2 },
  "selection": { "method": "uniform", "n_peaks": 30000 },
  "clustering": { "method": "graph_clustering", "k_nearest": 12 },
  "matching": { "method": "circus-omp-svd" },

  "merging": { "auto_merge_units": true, "acg_threshold": 0.2, "ccg_threshold": 0.25 },

  "motion_correction": { "enabled": false },

  "cache_preprocessing": { "mode": "memory", "memory_limit": "2G", "delete_cache": true },
  "chunk_preprocessing": { "n_jobs": 12, "total_memory": "4G" }
}""",
    "Global-ish": """{
  "apply_preprocessing": true,
  "apply_motion_correction": false,
  "matched_filtering": true,
  "templates_from_svd": true,
  "multi_units_only": false,
  "job_kwargs": { "n_jobs": 12 },
  "seed": 42,
  "deterministic_peaks_detection": true,

  "general": { "ms_before": 1.5, "ms_after": 2.5, "radius_um": 100000 },
  "sparsity": { "method": "radius", "radius_um": 100000 },

  "filtering": { "type": "butterworth", "high_pass": 300, "order": 3 },
  "whitening": { "mode": "global" },

  "detection": { "peak_sign": "both", "detect_threshold": 3.0, "min_peak_distance_ms": 1.2 },
  "selection": { "method": "uniform", "n_peaks": 35000 },
  "clustering": { "method": "graph_clustering", "k_nearest": 15 },
  "matching": { "method": "circus-omp-svd" },

  "merging": { "auto_merge_units": true, "acg_threshold": 0.2, "ccg_threshold": 0.25 },

  "motion_correction": { "enabled": false },

  "cache_preprocessing": { "mode": "memory", "memory_limit": "2G", "delete_cache": true },
  "chunk_preprocessing": { "n_jobs": 12, "total_memory": "4G" }
}"""
}

# Presets tuned for Nlx 32 kHz, tetrode, both signs
SC2_NLX_DEFAULT_JSON = """{
  "apply_preprocessing": false,
  "apply_motion_correction": false,
  "matched_filtering": true,
  "templates_from_svd": true,
  "multi_units_only": false,
  "job_kwargs": { "n_jobs": 12 },
  "seed": 42,
  "deterministic_peaks_detection": true,

  "general": { "ms_before": 1.5, "ms_after": 2.5, "radius_um": 80 },
  "sparsity": { "method": "radius", "radius_um": 80 },

  "whitening": { "mode": "local", "neighbors": 4 },

  "detection": { "peak_sign": "both", "detect_threshold": 4.0, "min_peak_distance_ms": 0.5 },
  "selection": { "method": "uniform", "n_peaks": 20000 },
  "clustering": { "method": "graph_clustering", "k_nearest": 4 },
  "matching": { "method": "circus-omp-svd" },

  "merging": { "auto_merge_units": true, "acg_threshold": 0.2, "ccg_threshold": 0.25 },

  "motion_correction": { "enabled": false },

  "cache_preprocessing": { "mode": "memory", "memory_limit": "2G", "delete_cache": true },
  "chunk_preprocessing": { "n_jobs": 12, "total_memory": "4G" }
}"""

sc2_preset_dd = w.Dropdown(
    options=["Local", "Wide", "Global-ish"],
    value="Local",
    description="SC2 preset",
    layout=w.Layout(width="240px")
)

sc2_params_json = w.Textarea(
    value="",
    placeholder="Edit SC2 preset JSON here…",
    description="Advanced JSON",
    layout=w.Layout(width="540px", height="220px")
)

def _fill_sc2_from_preset(change=None):
    if _get_mode() == "neuralynx_tetrode":
        sc2_params_json.value = SC2_NLX_DEFAULT_JSON
    else:
        sc2_params_json.value = sc2_presets[sc2_preset_dd.value]

def _apply_sc2_defaults_for_mode(force=False):
    is_nlx = (_get_mode() == "neuralynx_tetrode")
    sc2_preset_dd.layout.display = "none" if is_nlx else None
    if is_nlx:
        if force or not sc2_params_json.value.strip():
            sc2_params_json.value = SC2_NLX_DEFAULT_JSON
    elif force:
        sc2_params_json.value = sc2_presets[sc2_preset_dd.value]

_fill_sc2_from_preset()
sc2_preset_dd.observe(_fill_sc2_from_preset, "value")

sc2_params_box = w.VBox([sc2_info, sc2_preset_dd, sc2_params_json])
sc2_params_box.layout.display = "none"  # shown only when SC2 selected

# A stack that holds all sorter-specific panels; we will show/hide one at a time
param_stack = w.VBox([ms4_params_box, ks4_params_box, sc2_params_box])


sorter_explain_html = w.HTML("")

def _apply_all_sorter_defaults_for_mode(force=False):
    _apply_ms4_defaults_for_mode(force=force)
    _apply_ks4_defaults_for_mode(force=force)
    _apply_sc2_defaults_for_mode(force=force)

def _update_sorter_explanation(*args):
    mode = _get_mode()
    sorter = sorter_dd.value
    is_nlx = mode == "neuralynx_tetrode"
    mode_label = "Neuralynx tetrode (.ncs, 32 kHz, compact 4-contact probe)" if is_nlx else "MEA .h5 (20 kHz, geometry CSV)"

    if sorter == "mountainsort4":
        sorter_explain_html.value = f"""
        <div style='max-width:1100px; line-height:1.35;'>
        <b>Current input mode:</b> {mode_label}<br>
        <b>MountainSort4 parameters:</b>
        <ul>
          <li><b>detect_threshold</b>: threshold in estimated noise units. Higher is stricter; lower is more sensitive but can increase noise/splits.</li>
          <li><b>detect_sign</b>: -1 negative, +1 positive, 0 both polarities. For tetrode testing we use 0 initially.</li>
          <li><b>adjacency_radius</b>: spatial neighborhood in µm. In tetrode mode, 80 µm covers the compact 25 µm-square probe; local/wide/global presets are hidden.</li>
          <li><b>clip_size</b>: sorter waveform clip length in samples. At 32 kHz, 128 samples is ~4 ms, matching the original MEA time window.</li>
          <li><b>detect_interval</b>: duplicate-detection lockout in samples. At 32 kHz, 16 samples is ~0.5 ms.</li>
          <li><b>ms4 workers</b>: number of CPU workers used by MountainSort4.</li>
        </ul>
        </div>
        """
    elif sorter == "kilosort4":
        sorter_explain_html.value = f"""
        <div style='max-width:1100px; line-height:1.35;'>
        <b>Current input mode:</b> {mode_label}<br>
        <b>Kilosort4 advanced JSON:</b> edit the visible JSON above. In tetrode mode, local/wide/global presets are hidden and the JSON is prefilled for a 4-channel 32 kHz tetrode.
        <ul>
          <li><b>fs</b>: sampling rate passed to Kilosort4; tetrode default is 32000 Hz.</li>
          <li><b>highpass_cutoff</b>: Kilosort4 internal high-pass cutoff; tetrode default is 500 Hz.</li>
          <li><b>Th_universal / Th_learned</b>: detection/template thresholds; higher is more conservative.</li>
          <li><b>nt</b>: template/window length in samples; 129 samples at 32 kHz is ~4 ms.</li>
          <li><b>do_CAR</b>: common-average/reference step inside Kilosort4; for tetrodes this operates only across the retained tetrode channels.</li>
          <li><b>dmin, dminx, max_channel_distance</b>: geometry/neighborhood scale. Tetrode defaults use 25 µm contact spacing and 80 µm local neighborhood.</li>
          <li><b>whitening_range / nearest_chans</b>: nearby channels used for whitening/template neighborhoods; tetrode defaults are capped at 4 channels.</li>
          <li><b>duplicate_spike_ms</b>: temporal duplicate-removal window; tetrode default is 0.5 ms.</li>
          <li><b>cluster_neighbors / max_cluster_subset</b>: clustering neighborhood and maximum spikes used during clustering.</li>
          <li><b>skip_kilosort_preprocessing</b>: false means Kilosort4 handles its required preprocessing; the UI only applies optional notch to the raw KS4 input.</li>
        </ul>
        </div>
        """
    else:
        sorter_explain_html.value = f"""
        <div style='max-width:1100px; line-height:1.35;'>
        <b>Current input mode:</b> {mode_label}<br>
        <b>SpyKING Circus2 advanced JSON:</b> edit the visible JSON above. In tetrode mode, local/wide/global presets are hidden and the JSON is prefilled for a 4-channel 32 kHz tetrode.
        <ul>
          <li><b>apply_preprocessing</b>: in tetrode mode this is false because the UI already provides the bandpass/notch/CMR preprocessed recording to SC2.</li>
          <li><b>general.ms_before / general.ms_after</b>: waveform window around each spike used by SC2.</li>
          <li><b>general.radius_um / sparsity.radius_um</b>: spatial sparsity radius; tetrode default is 80 µm, which covers the compact 4-contact geometry.</li>
          <li><b>whitening.mode / whitening.neighbors</b>: local whitening settings; tetrode default uses up to 4 neighbors.</li>
          <li><b>detection.peak_sign</b>: spike polarity; "both" matches detect_sign=0 logic.</li>
          <li><b>detection.detect_threshold</b>: detection threshold in noise units; tetrode default starts at 4.0.</li>
          <li><b>detection.min_peak_distance_ms</b>: duplicate/minimum peak spacing; tetrode default is 0.5 ms.</li>
          <li><b>selection.n_peaks</b>: number of peaks sampled for clustering/template estimation.</li>
          <li><b>clustering.k_nearest</b>: nearest-neighbor count for graph clustering; tetrode default is 4.</li>
          <li><b>matching.method</b>: template matching method used after clustering.</li>
          <li><b>merging</b>: automatic merge thresholds; keep enabled for first pass, then review reports/QC.</li>
        </ul>
        </div>
        """

def _on_sorter_change(*args):
    is_ms4 = sorter_dd.value == "mountainsort4"
    is_ks4 = sorter_dd.value == "kilosort4"
    is_sc2 = sorter_dd.value == "spykingcircus2"

    # show only the active panel
    ms4_params_box.layout.display = None if is_ms4 else "none"
    ks4_params_box.layout.display = None if is_ks4 else "none"
    sc2_params_box.layout.display = None if is_sc2 else "none"

    # Apply mode-specific visibility/defaults for the preset controls.
    # In Neuralynx mode, preset dropdowns are hidden and tetrode-specific parameters are prefilled.
    _apply_all_sorter_defaults_for_mode(force=False)
    _update_sorter_explanation()

    # (optional) disable MS4 widgets when not in use
    for wid in (det_thr, det_sign, adj_rad, clip_size, det_int, ms4_workers):
        wid.disabled = not is_ms4

sorter_dd.observe(_on_sorter_change, "value")
_on_sorter_change()

run_sort_btn = w.Button(description="Run sorting", icon="cogs", button_style="warning")

# Metrics & reports
compute_metrics_btn   = w.Button(description="Compute metrics", icon="bar-chart", button_style="info")
export_spike_times_btn= w.Button(description="Export spike times", icon="file-excel-o")
build_reports_btn     = w.Button(description="Build unit reports", icon="file-image-o", button_style="success")

# ACG/report options and PC metrics
acg_toggle = w.Checkbox(value=True, description="Include ACGs (grid)")
acg_pdf    = w.Checkbox(value=True, description="Save ACGs as multi-page PDF")
pc_metrics = w.Checkbox(value=True, description="Include PC-based cluster metrics")

# --- NEW: compare sorters (MS4↔KS4↔SC2) ---
cmp_ms4_ks4_btn = w.Button(description="Compare MS4 vs KS4", icon="exchange", button_style="")
cmp_ms4_sc2_btn = w.Button(description="Compare MS4 vs SC2", icon="exchange", button_style="")
cmp_ks4_sc2_btn = w.Button(description="Compare KS4 vs SC2", icon="exchange", button_style="")

def _maybe_notch_raw_for_ks4(rec_raw, S, ref_rec_for_probe=None):
    if not getattr(S, "use_notch_60hz", False):
        return rec_raw
    try:
        import spikeinterface.preprocessing as spre
        rec_n = spre.notch_filter(rec_raw, freq=float(getattr(S, "hp_notch_hz", 60.0)))
        rec_n = core["ensure_probe_on"](rec_n, ref_rec_for_probe or rec_raw)
        _log(f"Applied {getattr(S, 'hp_notch_hz', 60.0):.1f} Hz notch on KS4 input.")
        return rec_n
    except Exception as e:
        _log(f"KS4 notch requested but failed ({e}); falling back to raw.")
        return rec_raw

def on_run_sort(_):
    log_out.clear_output()
    progress.value = 0
    if not core or state["preproc"] is None or state["qc_df"] is None or state["recording"] is None:
        _log("❗ Run QC first.")
        return

    # Sync QC edits and build include list
    _sync_qc_from_ui()
    inc = list(state["qc_df"].loc[state["qc_df"]["include"], "channel_id"].values)
    state["include_ch"] = inc

    # Prepare both QC and RAW subsets with identical channel masks
    try:
        sink = _ToWidget()  
        with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
            rec_qc  = core["subset_recording_channels"](state["preproc"],  inc) if inc else state["preproc"]
            rec_raw = core["subset_recording_channels"](state["recording"], inc) if inc else state["recording"]
            rec_qc  = core["ensure_probe_on"](rec_qc,  state["preproc"])
            rec_raw = core["ensure_probe_on"](rec_raw, state["recording"])
    except Exception as e:
        _log(f"Channel subset/ensure_probe error: {e}")
        return

    # Settings
    _ensure_settings()
    S = state["settings"]
    S.detect_threshold   = float(det_thr.value)
    S.detect_sign        = int(det_sign.value)
    S.adjacency_radius   = float(adj_rad.value)
    S.clip_size          = int(clip_size.value)
    S.detect_interval    = int(det_int.value)   
    try:
        fs_qc = float(rec_qc.get_sampling_frequency())
        _log(f"MS4 detect_interval: {S.detect_interval} samples (~{1000.0*S.detect_interval/fs_qc:.2f} ms)")
    except Exception:
        pass
    S.n_jobs             = int(ms4_workers.value)

    sorter = sorter_dd.value
    S.sorter_name = sorter

    # record current preset labels for provenance (even if panel hidden)
    S.ms4_preset = ms4_preset_dd.value
    S.ks4_preset = ks4_preset_dd.value
    S.sc2_preset = sc2_preset_dd.value

    # Capture optional per-sorter advanced params for core to consume
    S.ks4_params_json = ks4_params_json.value.strip() if sorter == "kilosort4" else ""
    S.sc2_params_json = sc2_params_json.value.strip() if sorter == "spykingcircus2" else ""

    import json
    if sorter == "kilosort4" and S.ks4_params_json:
        try: json.loads(S.ks4_params_json)
        except Exception as e:
            _log(f"KS4 advanced params JSON is invalid: {e}")
            return
    
    if sorter == "spykingcircus2" and S.sc2_params_json:
        try: json.loads(S.sc2_params_json)
        except Exception as e:
            _log(f"SC2 advanced params JSON is invalid: {e}")
            return

    rec_for_sort = (
        _maybe_notch_raw_for_ks4(rec_raw, S, ref_rec_for_probe=state["recording"])
        if sorter == "kilosort4" else rec_qc
    )
    rec_for_wave = rec_qc
    tag_map = {"mountainsort4":"ms4", "kilosort4":"ks4", "spykingcircus2":"sc2"}
    sort_out = str(Path(_get_active_run_dir(S.outdir))/f"si_sorting_{tag_map.get(sorter,'sort')}")

    # Run the selected sorter
    _log(f"Running {sorter.upper()}…")
    try:
        sink = _ToWidget()
        with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
            if sorter == "mountainsort4":
                state["sorting"] = core["run_mountainsort4"](rec_for_sort, S, sort_out)
            elif sorter == "kilosort4":
                state["sorting"] = core["run_kilosort4"](rec_for_sort, S, sort_out)
            elif sorter == "spykingcircus2":
                state["sorting"] = core["run_spykingcircus2"](rec_for_sort, S, sort_out)
            else:
                _log(f"Unknown sorter: {sorter}")
                return
            uids = state["sorting"].get_unit_ids()
            _log(f"Sorting done. Units found: {len(uids)}")
    except Exception as e:
        _log(f"Sorting error: {e}")
        return

    # Waveforms/PCA from the QC view
    _log("Extracting waveforms & PCA…")
    try:
        sink = _ToWidget()
        with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
            wf_dir = str(Path(_get_active_run_dir(S.outdir))/f"waveforms_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}")
            state["waveforms"] = core["extract_waveforms_and_pcs"](rec_for_wave, state["sorting"], S, wf_dir)
            _log("Waveforms/PCA ready.")
    except Exception as e:
        _log(f"Waveforms/PCA error: {e}")

run_sort_btn.on_click(on_run_sort)

def on_compute_metrics(_):
    log_out.clear_output()
    if state["waveforms"] is None:
        _log("Run sorting first.")
        return
    # Sync toggles into settings for report stage too
    _ensure_settings()
    S = state["settings"]
    S.include_acg_in_reports = bool(acg_toggle.value)
    S.acg_use_pdf            = bool(acg_pdf.value)
    S.compute_pc_metrics     = bool(pc_metrics.value)

    _log("Computing quality metrics…")
    try:
        sink = _ToWidget()  
        with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):  
            from spikeinterface import qualitymetrics as qim
            state["waveforms"].compute("templates")
            state["waveforms"].compute("spike_amplitudes")
            state["waveforms"].compute("noise_levels", method="mad")
            if S.compute_pc_metrics:
                state["waveforms"].compute("principal_components", n_components=3, mode="by_channel_global")

            state["metrics_df"] = qim.compute_quality_metrics(state["waveforms"], metric_names=None)

        out = str(Path(_get_active_run_dir(S.outdir))/"quality_metrics.xlsx")
        state["metrics_df"].to_excel(out)
        _log(f"Quality metrics written: {out}")
    except Exception as e:
        _log(f"Metrics error: {e}")

compute_metrics_btn.on_click(on_compute_metrics)

def on_export_spike_times(_):
    if state["sorting"] is None or state["waveforms"] is None:
        _log("❗ Run sorting first.")
        return
    try:
        fs = float(state["waveforms"].recording.get_sampling_frequency())
        out = str(Path(_get_active_run_dir(state["settings"].outdir))/"units_spike_times.xlsx")
        sink = _ToWidget()
        with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
            core["export_unit_spike_times"](state["sorting"], fs, out)
        _log(f"Spike times exported: {out}")
    except Exception as e:
        _log(f"Export error: {e}")

export_spike_times_btn.on_click(on_export_spike_times)

def on_build_reports(_):
    if state["waveforms"] is None:
        _log("Run sorting first.")
        return
    try:
        rep_dir = str(Path(_get_active_run_dir(state["settings"].outdir))/"unit_reports")
        # <<< NEW: capture all per-unit printouts >>>
        sink = _ToWidget()
        with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
            core["make_unit_report_figures"](state["waveforms"], rep_dir, state["settings"], state["metrics_df"])
        # >>> END NEW
        _log(f"Unit reports saved under: {rep_dir}")
    except Exception as e:
        _log(f"Report error: {e}")

build_reports_btn.on_click(on_build_reports)


# --- Configured sorter comparison workflow -----------------------------------------
# The comparison buttons do not immediately launch sorting. They open this configuration
# panel so the researcher can review/edit the two sorter parameter sets first.
cmp_panel_title = w.HTML("<b>Sorter comparison setup</b>")
cmp_pair_note = w.HTML("<em>Click one of the compare buttons to configure the sorter pair.</em>")
cmp_a_box = w.VBox([])
cmp_b_box = w.VBox([])
cmp_run_config_btn = w.Button(description="Run configured comparison", icon="play", button_style="warning")
cmp_cancel_btn = w.Button(description="Hide comparison setup", icon="times")
cmp_options = w.VBox([
    w.HTML("<b>Automatic outputs</b>"),
    w.HTML("<em>For each sorter, the comparison run automatically exports spike times, computes quality metrics, and builds unit reports. It then writes matched-pair outputs and an agreement heatmap.</em>"),
    w.HBox([cmp_run_config_btn, cmp_cancel_btn]),
])
cmp_config_panel = w.VBox([
    cmp_panel_title,
    cmp_pair_note,
    w.HBox([cmp_a_box, cmp_b_box]),
    cmp_options,
], layout=w.Layout(border="2px solid #999", padding="10px", margin="8px 0px 8px 0px"))
cmp_config_panel.layout.display = "none"

_COMPARE_SORTER_LABELS = {
    "mountainsort4": "MountainSort4",
    "kilosort4": "Kilosort4",
    "spykingcircus2": "SpyKING Circus2",
}
_COMPARE_TAG_MAP = {"mountainsort4":"ms4", "kilosort4":"ks4", "spykingcircus2":"sc2"}

def _json_pretty_or_raw(text):
    try:
        return json.dumps(json.loads(text), indent=2)
    except Exception:
        return text

def _current_ms4_param_dict():
    return dict(
        detect_threshold=float(det_thr.value),
        detect_sign=int(det_sign.value),
        adjacency_radius=float(adj_rad.value),
        clip_size=int(clip_size.value),
        detect_interval=int(det_int.value),
        n_jobs=int(ms4_workers.value),
        ms4_preset=ms4_preset_dd.value,
    )

def _default_ms4_param_dict_for_mode():
    if _get_mode() == "neuralynx_tetrode":
        return dict(
            detect_threshold=MS4_NLX_DEFAULTS["detect_threshold"],
            detect_sign=MS4_NLX_DEFAULTS["detect_sign"],
            adjacency_radius=MS4_NLX_DEFAULTS["adjacency_radius"],
            clip_size=MS4_NLX_DEFAULTS["clip_size"],
            detect_interval=MS4_NLX_DEFAULTS["detect_interval"],
            n_jobs=MS4_NLX_DEFAULTS["ms4_workers"],
            ms4_preset="Neuralynx tetrode direct defaults",
        )
    name = ms4_preset_dd.value
    radius = 200.0 if "Local" in name else (400.0 if "Wide" in name else -1.0)
    return dict(
        detect_threshold=MS4_MEA_DEFAULTS["detect_threshold"],
        detect_sign=MS4_MEA_DEFAULTS["detect_sign"],
        adjacency_radius=radius,
        clip_size=MS4_MEA_DEFAULTS["clip_size"],
        detect_interval=MS4_MEA_DEFAULTS["detect_interval"],
        n_jobs=MS4_MEA_DEFAULTS["ms4_workers"],
        ms4_preset=name,
    )

def _default_json_for_sorter(sorter_name):
    if sorter_name == "kilosort4":
        if _get_mode() == "neuralynx_tetrode":
            return KS4_NLX_DEFAULT_JSON
        return ks4_presets.get(ks4_preset_dd.value, ks4_params_json.value)
    if sorter_name == "spykingcircus2":
        if _get_mode() == "neuralynx_tetrode":
            return SC2_NLX_DEFAULT_JSON
        return sc2_presets.get(sc2_preset_dd.value, sc2_params_json.value)
    return ""

def _current_json_for_sorter(sorter_name):
    if sorter_name == "kilosort4":
        return ks4_params_json.value.strip() or _default_json_for_sorter(sorter_name)
    if sorter_name == "spykingcircus2":
        return sc2_params_json.value.strip() or _default_json_for_sorter(sorter_name)
    return ""

def _make_compare_sorter_panel(sorter_name, side_label):
    """Return (panel_widget, getter_function) for one side of the comparison."""
    nice = _COMPARE_SORTER_LABELS.get(sorter_name, sorter_name)
    mode_label = "Neuralynx tetrode" if _get_mode() == "neuralynx_tetrode" else "MEA .h5"

    if sorter_name == "mountainsort4":
        defaults = _default_ms4_param_dict_for_mode()
        current = _current_ms4_param_dict()
        # Start from current visible values, but keep a reset button to restore mode defaults.
        cmp_det_thr = w.BoundedFloatText(value=current["detect_threshold"], min=1.0, step=0.1, description="detect_threshold", layout=w.Layout(width="220px"))
        cmp_det_sign = w.IntText(value=current["detect_sign"], description="detect_sign", layout=w.Layout(width="220px"))
        cmp_adj = w.BoundedFloatText(value=current["adjacency_radius"], min=-1.0, max=100000.0, step=5.0, description="adjacency_radius", layout=w.Layout(width="220px"))
        cmp_clip = w.BoundedIntText(value=current["clip_size"], min=20, max=512, step=2, description="clip_size", layout=w.Layout(width="220px"))
        cmp_int = w.BoundedIntText(value=current["detect_interval"], min=1, max=400, description="detect_interval", layout=w.Layout(width="220px"))
        cmp_jobs = w.BoundedIntText(value=current["n_jobs"], min=1, max=os.cpu_count() or 256, description="workers", layout=w.Layout(width="220px"))
        reset_btn = w.Button(description="Reset MS4 defaults", icon="refresh")
        explain = w.HTML(f"""
        <div style='max-width:520px; line-height:1.35;'>
        <b>{side_label}: {nice}</b><br>
        <b>Mode:</b> {mode_label}<br>
        <ul>
          <li><b>detect_threshold</b>: MS4 detection sensitivity in its internal noise-normalized/whitened space. Higher is stricter.</li>
          <li><b>detect_sign</b>: -1 negative, +1 positive, 0 both polarities.</li>
          <li><b>adjacency_radius</b>: spatial neighborhood in µm. For Nlx tetrodes, 80 µm covers the compact 25 µm-square geometry.</li>
          <li><b>clip_size</b>: sorter clip length in samples.</li>
          <li><b>detect_interval</b>: duplicate-detection lockout in samples.</li>
          <li><b>workers</b>: CPU workers used by MS4.</li>
        </ul>
        </div>
        """)
        def _reset(_=None):
            cmp_det_thr.value = float(defaults["detect_threshold"])
            cmp_det_sign.value = int(defaults["detect_sign"])
            cmp_adj.value = float(defaults["adjacency_radius"])
            cmp_clip.value = int(defaults["clip_size"])
            cmp_int.value = int(defaults["detect_interval"])
            cmp_jobs.value = int(defaults["n_jobs"])
        reset_btn.on_click(_reset)
        panel = w.VBox([explain, w.HBox([cmp_det_thr, cmp_det_sign]), w.HBox([cmp_adj, cmp_clip]), w.HBox([cmp_int, cmp_jobs]), reset_btn], layout=w.Layout(border="1px solid #ddd", padding="8px", width="48%"))
        def getter():
            return dict(
                sorter_name=sorter_name,
                detect_threshold=float(cmp_det_thr.value),
                detect_sign=int(cmp_det_sign.value),
                adjacency_radius=float(cmp_adj.value),
                clip_size=int(cmp_clip.value),
                detect_interval=int(cmp_int.value),
                n_jobs=int(cmp_jobs.value),
                ms4_preset=defaults.get("ms4_preset", "comparison_custom"),
                ks4_params_json="",
                sc2_params_json="",
            )
        return panel, getter

    # KS4 / SC2 JSON panel
    current_json = _json_pretty_or_raw(_current_json_for_sorter(sorter_name))
    default_json = _json_pretty_or_raw(_default_json_for_sorter(sorter_name))
    text_height = "260px" if sorter_name == "spykingcircus2" else "220px"
    cmp_json = w.Textarea(value=current_json, description="JSON", layout=w.Layout(width="520px", height=text_height))
    reset_btn = w.Button(description=f"Reset {nice} defaults", icon="refresh")
    if sorter_name == "kilosort4":
        details = "Kilosort4 receives the raw included-channel recording with optional notch, then performs its native preprocessing/whitening/template detection."
        bullets = "fs, highpass_cutoff, Th_universal/Th_learned, nt, do_CAR, geometry/neighborhood settings, duplicate_spike_ms, and cluster settings."
    else:
        details = "SpyKING Circus2 receives the Tab 1 preprocessed included-channel recording in this GUI workflow."
        bullets = "apply_preprocessing, ms_before/ms_after, sparsity radius, whitening, detection threshold/sign, min peak distance, selection, clustering, matching, and merging."
    explain = w.HTML(f"""
    <div style='max-width:540px; line-height:1.35;'>
    <b>{side_label}: {nice}</b><br>
    <b>Mode:</b> {mode_label}<br>
    {details}<br>
    <b>Editable JSON includes:</b> {bullets}<br>
    The JSON is validated before the comparison run starts.
    </div>
    """)
    def _reset(_=None):
        cmp_json.value = default_json
    reset_btn.on_click(_reset)
    panel = w.VBox([explain, cmp_json, reset_btn], layout=w.Layout(border="1px solid #ddd", padding="8px", width="48%"))
    def getter():
        raw = cmp_json.value.strip()
        if raw:
            # validate and normalize spacing
            raw = json.dumps(json.loads(raw), indent=2)
        return dict(
            sorter_name=sorter_name,
            detect_threshold=float(det_thr.value),
            detect_sign=int(det_sign.value),
            adjacency_radius=float(adj_rad.value),
            clip_size=int(clip_size.value),
            detect_interval=int(det_int.value),
            n_jobs=int(ms4_workers.value),
            ms4_preset=ms4_preset_dd.value,
            ks4_params_json=raw if sorter_name == "kilosort4" else "",
            sc2_params_json=raw if sorter_name == "spykingcircus2" else "",
        )
    return panel, getter

_cmp_getter_a = None
_cmp_getter_b = None

def _show_compare_config(name_a: str, name_b: str):
    """Open the comparison configuration panel for the selected pair."""
    global _cmp_getter_a, _cmp_getter_b
    if not core:
        _log("Load the core module first.")
        return
    label_a = _COMPARE_SORTER_LABELS.get(name_a, name_a)
    label_b = _COMPARE_SORTER_LABELS.get(name_b, name_b)
    panel_a, getter_a = _make_compare_sorter_panel(name_a, "Sorter A")
    panel_b, getter_b = _make_compare_sorter_panel(name_b, "Sorter B")
    _cmp_getter_a = getter_a
    _cmp_getter_b = getter_b
    state["compare_pair"] = (name_a, name_b)
    cmp_panel_title.value = f"<b>Sorter comparison setup: {label_a} vs {label_b}</b>"
    cmp_pair_note.value = "<em>Review or edit both parameter sets, then click <b>Run configured comparison</b>. The same included-channel mask from Tab 1 will be used for both sorters.</em>"
    cmp_a_box.children = [panel_a]
    cmp_b_box.children = [panel_b]
    cmp_config_panel.layout.display = None
    _log(f"Opened comparison setup for {label_a} vs {label_b}.")

def _settings_from_compare_params(params, base_settings):
    """Create an independent MEASettings instance for one comparison sorter."""
    import copy
    S = copy.copy(base_settings)
    S.sorter_name = params["sorter_name"]
    S.detect_threshold = float(params.get("detect_threshold", det_thr.value))
    S.detect_sign = int(params.get("detect_sign", det_sign.value))
    S.adjacency_radius = float(params.get("adjacency_radius", adj_rad.value))
    S.clip_size = int(params.get("clip_size", clip_size.value))
    S.detect_interval = int(params.get("detect_interval", det_int.value))
    S.n_jobs = int(params.get("n_jobs", ms4_workers.value))
    S.ms4_preset = params.get("ms4_preset", ms4_preset_dd.value)
    S.ks4_preset = ks4_preset_dd.value
    S.sc2_preset = sc2_preset_dd.value
    S.ks4_params_json = params.get("ks4_params_json", "") if S.sorter_name == "kilosort4" else ""
    S.sc2_params_json = params.get("sc2_params_json", "") if S.sorter_name == "spykingcircus2" else ""
    S.include_acg_in_reports = bool(acg_toggle.value)
    S.acg_use_pdf = bool(acg_pdf.value)
    S.compute_pc_metrics = bool(pc_metrics.value)
    return S

def _write_comparison_manifest(out_dir, rows):
    try:
        pd.DataFrame(rows).to_csv(Path(out_dir) / "comparison_output_manifest.csv", index=False)
    except Exception as e:
        _log(f"Could not write comparison manifest CSV: {e}")

def _run_sorter_with_automatic_outputs(sorter_name, S, rec_for_sort, rec_for_wave, sorter_dir):
    """
    Run one sorter for comparison and automatically generate the same deliverables
    normally produced by Tab 2 buttons: spike times, quality metrics, and reports.
    """
    tag = _COMPARE_TAG_MAP.get(sorter_name, "sort")
    sorter_dir = Path(sorter_dir)
    sorter_dir.mkdir(parents=True, exist_ok=True)
    sorting_folder = str(sorter_dir / "si_sorting")
    waveforms_folder = str(sorter_dir / "waveforms")
    spike_times_path = str(sorter_dir / "units_spike_times.xlsx")
    metrics_path = str(sorter_dir / "quality_metrics.xlsx")
    reports_dir = str(sorter_dir / "unit_reports")
    settings_path = sorter_dir / "sorter_settings_used.json"

    # Save settings used for reproducibility.
    try:
        settings_payload = dict(
            sorter_name=sorter_name,
            detect_threshold=getattr(S, "detect_threshold", None),
            detect_sign=getattr(S, "detect_sign", None),
            adjacency_radius=getattr(S, "adjacency_radius", None),
            clip_size=getattr(S, "clip_size", None),
            detect_interval=getattr(S, "detect_interval", None),
            n_jobs=getattr(S, "n_jobs", None),
            ks4_params_json=getattr(S, "ks4_params_json", ""),
            sc2_params_json=getattr(S, "sc2_params_json", ""),
            include_acg_in_reports=getattr(S, "include_acg_in_reports", None),
            acg_use_pdf=getattr(S, "acg_use_pdf", None),
            compute_pc_metrics=getattr(S, "compute_pc_metrics", None),
        )
        settings_path.write_text(json.dumps(settings_payload, indent=2))
    except Exception as e:
        _log(f"Could not save settings for {sorter_name}: {e}")

    _log(f"[{tag}] Running sorter: {sorter_name}…")
    sink = _ToWidget()
    with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
        if sorter_name == "mountainsort4":
            sorting = core["run_mountainsort4"](rec_for_sort, S, sorting_folder)
        elif sorter_name == "kilosort4":
            sorting = core["run_kilosort4"](rec_for_sort, S, sorting_folder)
        elif sorter_name == "spykingcircus2":
            sorting = core["run_spykingcircus2"](rec_for_sort, S, sorting_folder)
        else:
            raise ValueError(f"Unknown sorter: {sorter_name}")
    n_units = len(sorting.get_unit_ids())
    _log(f"[{tag}] Sorting complete. Units found: {n_units}")

    # 1) Export spike times immediately after sorting.
    _log(f"[{tag}] Exporting spike times…")
    fs = float(rec_for_wave.get_sampling_frequency())
    sink = _ToWidget()
    with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
        core["export_unit_spike_times"](sorting, fs, spike_times_path)
    _log(f"[{tag}] Spike times: {spike_times_path}")

    # Internal analyzer extraction needed before metrics/reports.
    _log(f"[{tag}] Extracting waveforms/PCA for metrics and reports…")
    sink = _ToWidget()
    with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
        waveforms = core["extract_waveforms_and_pcs"](rec_for_wave, sorting, S, waveforms_folder)
    _log(f"[{tag}] Waveforms/analyzer folder: {waveforms_folder}")

    # 2) Compute metrics.
    _log(f"[{tag}] Computing quality metrics…")
    sink = _ToWidget()
    with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
        from spikeinterface import qualitymetrics as qim
        waveforms.compute("templates")
        waveforms.compute("spike_amplitudes")
        waveforms.compute("noise_levels", method="mad")
        if getattr(S, "compute_pc_metrics", False):
            waveforms.compute("principal_components", n_components=3, mode="by_channel_global")
        metrics_df = qim.compute_quality_metrics(waveforms, metric_names=None)
    metrics_df.to_excel(metrics_path)
    _log(f"[{tag}] Quality metrics: {metrics_path}")

    # 3) Build reports.
    _log(f"[{tag}] Building unit reports…")
    sink = _ToWidget()
    with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
        core["make_unit_report_figures"](waveforms, reports_dir, S, metrics_df)
    _log(f"[{tag}] Unit reports: {reports_dir}")

    return dict(
        sorter_name=sorter_name,
        tag=tag,
        sorting=sorting,
        waveforms=waveforms,
        metrics_df=metrics_df,
        n_units=n_units,
        sorter_dir=str(sorter_dir),
        sorting_folder=sorting_folder,
        waveforms_folder=waveforms_folder,
        spike_times_path=spike_times_path,
        metrics_path=metrics_path,
        reports_dir=reports_dir,
        settings_path=str(settings_path),
    )

def _run_configured_comparison(_=None):
    """Run the currently configured comparison pair and write all comparison outputs."""
    import spikeinterface.comparison as sicomp
    import matplotlib.pyplot as plt

    if state.get("recording") is None or state.get("preproc") is None or state.get("qc_df") is None:
        _log("❗ Run QC first.")
        return
    if _cmp_getter_a is None or _cmp_getter_b is None or "compare_pair" not in state:
        _log("Choose a sorter comparison first.")
        return

    log_out.clear_output()
    progress.value = 0

    try:
        params_a = _cmp_getter_a()
        params_b = _cmp_getter_b()
    except Exception as e:
        _log(f"Comparison parameter error: {e}")
        return

    name_a, name_b = params_a["sorter_name"], params_b["sorter_name"]
    tag_a, tag_b = _COMPARE_TAG_MAP.get(name_a, "a"), _COMPARE_TAG_MAP.get(name_b, "b")
    label_a, label_b = _COMPARE_SORTER_LABELS.get(name_a, name_a), _COMPARE_SORTER_LABELS.get(name_b, name_b)

    # Sync per-channel selection from Tab 1.
    _sync_qc_from_ui()
    inc = list(state["qc_df"].loc[state["qc_df"]["include"], "channel_id"].values)
    state["include_ch"] = inc

    try:
        sink = _ToWidget()
        with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
            rec_qc  = core["subset_recording_channels"](state["preproc"],  inc) if inc else state["preproc"]
            rec_raw = core["subset_recording_channels"](state["recording"], inc) if inc else state["recording"]
            rec_qc  = core["ensure_probe_on"](rec_qc,  state["preproc"])
            rec_raw = core["ensure_probe_on"](rec_raw, state["recording"])
    except Exception as e:
        _log(f"Channel subset/ensure_probe error: {e}")
        return

    _ensure_settings()
    base_S = state["settings"]
    base_out = Path(_get_active_run_dir(base_S.outdir))
    compare_out = base_out / f"sorter_comparison_{tag_a}_vs_{tag_b}_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
    compare_out.mkdir(parents=True, exist_ok=True)

    S_a = _settings_from_compare_params(params_a, base_S)
    S_b = _settings_from_compare_params(params_b, base_S)
    S_a.outdir = str(compare_out / f"{tag_a}_{name_a}")
    S_b.outdir = str(compare_out / f"{tag_b}_{name_b}")

    def _rec_for(sorter_name, S):
        if sorter_name == "kilosort4":
            return _maybe_notch_raw_for_ks4(rec_raw, S, ref_rec_for_probe=state["recording"])
        return rec_qc

    _log(f"Comparison output folder: {compare_out}")
    _log(f"Running configured comparison: {label_a} vs {label_b}")

    try:
        result_a = _run_sorter_with_automatic_outputs(
            name_a, S_a, _rec_for(name_a, S_a), rec_qc, compare_out / f"{tag_a}_{name_a}"
        )
    except Exception as e:
        _log(f"{label_a} comparison run failed: {e}")
        return

    try:
        result_b = _run_sorter_with_automatic_outputs(
            name_b, S_b, _rec_for(name_b, S_b), rec_qc, compare_out / f"{tag_b}_{name_b}"
        )
    except Exception as e:
        _log(f"{label_b} comparison run failed: {e}")
        return

    # Compare sorter outputs.
    _log("Computing sorter agreement/matching…")
    comp = sicomp.compare_two_sorters(
        sorting1=result_a["sorting"], sorting2=result_b["sorting"],
        sorting1_name=label_a, sorting2_name=label_b
    )

    try:
        df_pairs = comp.get_matching().copy()
    except AttributeError:
        df_pairs = comp.get_best_match().copy()

    rename_map = {}
    if "unit_id1" in df_pairs.columns: rename_map["unit_id1"] = f"{tag_a}_unit_id"
    if "unit_id2" in df_pairs.columns: rename_map["unit_id2"] = f"{tag_b}_unit_id"
    if "match_score" in df_pairs.columns: rename_map["match_score"] = "agreement"
    if "agreement_score" in df_pairs.columns: rename_map["agreement_score"] = "agreement"
    df_pairs = df_pairs.rename(columns=rename_map)

    # Unmatched units.
    get_un1 = getattr(comp, "get_unmatched1", getattr(comp, "get_non_matched1", None))
    get_un2 = getattr(comp, "get_unmatched2", getattr(comp, "get_non_matched2", None))
    un1 = list(get_un1()) if callable(get_un1) else []
    un2 = list(get_un2()) if callable(get_un2) else []
    df_un1 = pd.DataFrame({f"unmatched_{tag_a}_unit_id": un1})
    df_un2 = pd.DataFrame({f"unmatched_{tag_b}_unit_id": un2})

    units_a = list(result_a["sorting"].get_unit_ids())
    units_b = list(result_b["sorting"].get_unit_ids())
    idx_a = {u: i for i, u in enumerate(units_a)}
    idx_b = {u: i for i, u in enumerate(units_b)}
    M = np.zeros((len(units_a), len(units_b)), dtype=float)
    has_agreement = "agreement" in df_pairs.columns
    for _, row in df_pairs.iterrows():
        ua = row.get(f"{tag_a}_unit_id")
        ub = row.get(f"{tag_b}_unit_id")
        if (ua in idx_a) and (ub in idx_b):
            M[idx_a[ua], idx_b[ub]] = float(row["agreement"]) if has_agreement else 1.0

    matrix_df = pd.DataFrame(M, index=[str(u) for u in units_a], columns=[str(u) for u in units_b])
    matrix_df.index.name = f"{tag_a}_unit_id"

    pairs_csv = compare_out / f"{tag_a}_vs_{tag_b}_matched_pairs.csv"
    un_a_csv = compare_out / f"unmatched_{tag_a}.csv"
    un_b_csv = compare_out / f"unmatched_{tag_b}.csv"
    matrix_csv = compare_out / f"agreement_matrix_{tag_a}_vs_{tag_b}.csv"
    summary_xlsx = compare_out / f"comparison_summary_{tag_a}_vs_{tag_b}.xlsx"
    heat_path = compare_out / f"agreement_heatmap_{tag_a}_vs_{tag_b}.png"
    manifest_csv = compare_out / "comparison_output_manifest.csv"

    df_pairs.to_csv(pairs_csv, index=False)
    df_un1.to_csv(un_a_csv, index=False)
    df_un2.to_csv(un_b_csv, index=False)
    matrix_df.to_csv(matrix_csv)

    summary_df = pd.DataFrame([
        {"field": "sorter_a", "value": label_a},
        {"field": "sorter_b", "value": label_b},
        {"field": f"{tag_a}_units", "value": result_a["n_units"]},
        {"field": f"{tag_b}_units", "value": result_b["n_units"]},
        {"field": "matched_pairs", "value": len(df_pairs)},
        {"field": f"unmatched_{tag_a}", "value": len(un1)},
        {"field": f"unmatched_{tag_b}", "value": len(un2)},
        {"field": "output_folder", "value": str(compare_out)},
    ])
    manifest_rows = [
        {"output": "comparison_root", "path": str(compare_out), "description": "Root folder for this sorter-comparison run."},
        {"output": f"{tag_a}_sorter_subfolder", "path": result_a["sorter_dir"], "description": f"All automatic outputs for {label_a}."},
        {"output": f"{tag_b}_sorter_subfolder", "path": result_b["sorter_dir"], "description": f"All automatic outputs for {label_b}."},
        {"output": f"{tag_a}_spike_times", "path": result_a["spike_times_path"], "description": f"Spike times for each {label_a} unit."},
        {"output": f"{tag_b}_spike_times", "path": result_b["spike_times_path"], "description": f"Spike times for each {label_b} unit."},
        {"output": f"{tag_a}_quality_metrics", "path": result_a["metrics_path"], "description": f"Quality metrics for {label_a} units."},
        {"output": f"{tag_b}_quality_metrics", "path": result_b["metrics_path"], "description": f"Quality metrics for {label_b} units."},
        {"output": f"{tag_a}_unit_reports", "path": result_a["reports_dir"], "description": f"Per-unit figures/reports for {label_a}."},
        {"output": f"{tag_b}_unit_reports", "path": result_b["reports_dir"], "description": f"Per-unit figures/reports for {label_b}."},
        {"output": "matched_pairs_csv", "path": str(pairs_csv), "description": "Matched units and agreement scores between the two sorters."},
        {"output": f"unmatched_{tag_a}", "path": str(un_a_csv), "description": f"Units found by {label_a} without a matched partner in {label_b}."},
        {"output": f"unmatched_{tag_b}", "path": str(un_b_csv), "description": f"Units found by {label_b} without a matched partner in {label_a}."},
        {"output": "agreement_matrix_csv", "path": str(matrix_csv), "description": "Unit-by-unit agreement matrix used to draw the heatmap."},
        {"output": "agreement_heatmap_png", "path": str(heat_path), "description": "Visual heatmap of agreement scores between units from both sorters."},
        {"output": "comparison_summary_xlsx", "path": str(summary_xlsx), "description": "Workbook collecting summary, matches, unmatched units, matrix, and manifest."},
    ]

    # Heatmap figure.
    if len(units_a) == 0 or len(units_b) == 0:
        _log("No units for one or both sorters; skipping heatmap.")
    else:
        fig, ax = plt.subplots(
            figsize=(max(4, len(units_b) * 0.25), max(3, len(units_a) * 0.25)),
            dpi=150
        )
        im = ax.imshow(M, aspect="auto", interpolation="nearest", vmin=0.0, vmax=1.0)
        ax.set_title(f"Agreement: {label_a} vs {label_b}")
        ax.set_xlabel(f"{label_b} units"); ax.set_ylabel(f"{label_a} units")
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Agreement (0–1)")
        if len(units_a) <= 50:
            ax.set_yticks(range(len(units_a)))
            ax.set_yticklabels([str(u) for u in units_a], fontsize=6)
        else:
            ax.set_yticks([])
        if len(units_b) <= 50:
            ax.set_xticks(range(len(units_b)))
            ax.set_xticklabels([str(u) for u in units_b], fontsize=6, rotation=90)
        else:
            ax.set_xticks([])
        fig.tight_layout()
        fig.savefig(heat_path)
        plt.close(fig)
        _log(f"Agreement heatmap: {heat_path}")

    _write_comparison_manifest(compare_out, manifest_rows)
    try:
        with pd.ExcelWriter(summary_xlsx, engine="xlsxwriter") as xl:
            summary_df.to_excel(xl, sheet_name="run_summary", index=False)
            pd.DataFrame(manifest_rows).to_excel(xl, sheet_name="output_manifest", index=False)
            df_pairs.to_excel(xl, sheet_name="matched_pairs", index=False)
            df_un1.to_excel(xl, sheet_name=f"unmatched_{tag_a}", index=False)
            df_un2.to_excel(xl, sheet_name=f"unmatched_{tag_b}", index=False)
            matrix_df.to_excel(xl, sheet_name="agreement_matrix")
    except Exception as e:
        _log(f"Could not write comparison summary workbook: {e}")

    _log(f"Matched pairs CSV: {pairs_csv}")
    _log(f"Unmatched {label_a}: {un_a_csv}")
    _log(f"Unmatched {label_b}: {un_b_csv}")
    _log(f"Agreement matrix CSV: {matrix_csv}")
    _log(f"Comparison summary workbook: {summary_xlsx}")
    _log(f"Comparison manifest: {manifest_csv}")
    _log("Configured sorter comparison complete.")

    # Show heatmap inline if available.
    if heat_path.exists():
        try:
            from IPython.display import Image, display
            with open(heat_path, "rb") as f:
                display(Image(data=f.read()))
        except Exception:
            pass


def on_cmp_ms4_ks4(_): _show_compare_config("mountainsort4", "kilosort4")
def on_cmp_ms4_sc2(_): _show_compare_config("mountainsort4", "spykingcircus2")
def on_cmp_ks4_sc2(_): _show_compare_config("kilosort4", "spykingcircus2")

def on_cmp_cancel(_):
    cmp_config_panel.layout.display = "none"

cmp_ms4_ks4_btn.on_click(on_cmp_ms4_ks4)
cmp_ms4_sc2_btn.on_click(on_cmp_ms4_sc2)
cmp_ks4_sc2_btn.on_click(on_cmp_ks4_sc2)
cmp_run_config_btn.on_click(_run_configured_comparison)
cmp_cancel_btn.on_click(on_cmp_cancel)


# =========================== TAB 4: Unit Curation ===================================
cur_metrics_path = w.Text(
    value="",
    placeholder="Path to quality_metrics.xlsx from Tab 2",
    description="Metrics file:",
    layout=w.Layout(width="70%")
)
cur_find_metrics_btn = w.Button(description="Find quality_metrics", icon="search")
cur_output_name = w.Text(value="curated_units_two_pass.xlsx", description="Output file:", layout=w.Layout(width="380px"))
cur_amp_max = w.BoundedFloatText(value=0.10, min=0.0, max=1.0, step=0.01, description="amp cutoff ≤", layout=w.Layout(width="190px"))
cur_presence_min = w.BoundedFloatText(value=0.90, min=0.0, max=1.0, step=0.01, description="presence ≥", layout=w.Layout(width="190px"))
cur_contam_max = w.BoundedFloatText(value=0.50, min=0.0, max=100.0, step=0.01, description="contam ≤", layout=w.Layout(width="190px"))
cur_nn_min = w.BoundedFloatText(value=0.70, min=0.0, max=1.0, step=0.01, description="NN hit ≥", layout=w.Layout(width="190px"))
cur_apply_nn = w.Checkbox(value=True, description="Apply NN-hit-rate pass when column exists")
cur_amp_nan_keep = w.Checkbox(value=True, description="Treat NaN amplitude_cutoff as keep")
cur_nn_nan_keep = w.Checkbox(value=True, description="Treat missing/NaN nn_hit_rate as keep")
cur_require_nn = w.Checkbox(value=False, description="Error if NN column is missing")
cur_run_btn = w.Button(description="Run two-pass curation", icon="filter", button_style="success")
cur_summary_html = w.HTML("<em>Curation summary will appear here.</em>")
cur_explain_html = w.HTML("""
<div style='line-height:1.35'>
<b>How Tab 4 works</b><br>
Run Tab 2 sorting and quality metrics first. Tab 4 reads <code>quality_metrics.xlsx</code>, fixes the unit-ID column if it was saved as the Excel index, and applies both curation passes in one step.
<ul>
  <li><b>amplitude_cutoff ≤ 0.10</b>: flags units with poor amplitude completeness. Keeping NaN amplitude_cutoff is useful because this metric can fail or be undefined even when other metrics look valid; the output still records <code>cur_amp_cutoff_is_nan</code>.</li>
  <li><b>presence_ratio ≥ 0.90</b>: favors units present across most of the recording. This is strong for stable tetrode single units; lower it only if the design has true state-dependent firing or shorter valid epochs.</li>
  <li><b>contamination ≤ 0.50</b>: uses <code>isi_violations</code>, then <code>isi_violations_ratio</code>, then <code>rp_contamination</code>. This is an Allen-style permissive contamination gate; tighten after visual review for very strict final single-unit sets.</li>
  <li><b>NN hit rate ≥ 0.70</b>: second-pass cluster-separation/isolation gate. This is reasonable as an automated screen for tetrodes, but 0.80–0.90 is stricter. If PC/NN metrics were not computed, the default is to keep rather than drop solely because the column is absent.</li>
</ul>
Output workbook sheets: <code>combined_units</code>, <code>kept_only</code>, <code>flagged_only</code>, <code>curation_summary</code>, and <code>curation_settings</code>.
</div>
""")

def _find_quality_metrics_file():
    base = Path(_get_active_run_dir(outdir_tb.value.strip() or str(Path.cwd())))
    direct = base / "quality_metrics.xlsx"
    if direct.exists():
        return direct
    root = Path(outdir_tb.value.strip() or str(Path.cwd()))
    candidates = list(root.rglob("quality_metrics.xlsx")) if root.exists() else []
    if candidates:
        return sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    return direct

def on_find_quality_metrics(_):
    p = _find_quality_metrics_file()
    cur_metrics_path.value = str(p)
    _log(f"{'Found' if p.exists() else 'Expected'} quality metrics: {p}")
cur_find_metrics_btn.on_click(on_find_quality_metrics)

def on_run_curation(_):
    log_out.clear_output()
    if not core:
        _log("Load the core module first.")
        return
    if "curate_quality_metrics_two_pass" not in core:
        _log("Loaded core does not contain curate_quality_metrics_two_pass. Load the patched core with Tab 4 support.")
        return
    metrics_path = (cur_metrics_path.value or "").strip() or str(_find_quality_metrics_file())
    cur_metrics_path.value = metrics_path
    p = Path(metrics_path)
    if not p.exists():
        _log(f"Quality metrics file not found: {p}")
        _log("Run Tab 2 → Compute metrics first, or point Tab 4 to an existing quality_metrics.xlsx/csv file.")
        return
    out_path = Path(cur_output_name.value.strip() or "curated_units_two_pass.xlsx")
    if not out_path.is_absolute():
        out_path = p.parent / out_path
    _log("Running two-pass unit curation…")
    try:
        sink = _ToWidget()
        with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
            saved_path, summary = core["curate_quality_metrics_two_pass"](
                metrics_path=str(p),
                output_path=str(out_path),
                amp_cutoff_max=float(cur_amp_max.value),
                presence_min=float(cur_presence_min.value),
                contamination_max=float(cur_contam_max.value),
                nn_hit_rate_min=float(cur_nn_min.value),
                apply_nn_gate=bool(cur_apply_nn.value),
                treat_amp_nan_as_keep=bool(cur_amp_nan_keep.value),
                treat_missing_nn_as_keep=bool(cur_nn_nan_keep.value),
                require_nn_column=bool(cur_require_nn.value),
            )
        _log(f"Curated workbook saved: {saved_path}")
        cur_summary_html.value = f"""
        <b>Two-pass curation complete</b><br>
        Output: <code>{saved_path}</code><br>
        Total units: <b>{summary.get('total_units')}</b> &nbsp;|&nbsp;
        Kept: <b>{summary.get('kept_units')}</b> &nbsp;|&nbsp;
        Flagged: <b>{summary.get('flagged_units')}</b> &nbsp;|&nbsp;
        Kept: <b>{summary.get('kept_percent', float('nan')):.1f}%</b><br>
        Contamination column: <code>{summary.get('contamination_column')}</code>
        ({summary.get('contamination_interpretation')})<br>
        NN column: <code>{summary.get('nn_column')}</code> &nbsp;|&nbsp;
        NN gate applied: <b>{summary.get('nn_gate_applied')}</b>
        """
    except Exception as e:
        _log(f"Curation error: {e}")
        cur_summary_html.value = f"<span style='color:#b00'>Curation error: {e}</span>"
cur_run_btn.on_click(on_run_curation)

# =========================== TAB 5: Export/Batch ====================================
animal_tb = w.Text(value="", description="Animal", layout=w.Layout(width="220px"))
uid_tb    = w.Text(value="", description="UniqueID", layout=w.Layout(width="220px"))
group_tb  = w.Text(value="", description="Group", layout=w.Layout(width="220px"))
cohort_tb = w.Text(value="", description="Cohort", layout=w.Layout(width="220px"))
save_meta_btn = w.Button(description="Save metadata", icon="save")

def on_save_meta(_):
    _ensure_settings()
    S = state["settings"]
    S.animal    = animal_tb.value
    S.unique_id = uid_tb.value
    S.group     = group_tb.value
    S.cohort    = cohort_tb.value
    d = dict(
        animal=S.animal, unique_id=S.unique_id, group=S.group, cohort=S.cohort,
        outdir=_get_active_run_dir(S.outdir),
        input_mode=getattr(S, "input_mode", _get_mode()),
        input_path=getattr(S, "input_path", _get_input_path()),
        selected_tetrode=getattr(S, "selected_tetrode", state.get("selected_tetrode", "")),
        sorter=getattr(S, "sorter_name", sorter_dd.value),
        ks4_params_json=getattr(S, "ks4_params_json", ""),
        sc2_params_json=getattr(S, "sc2_params_json", ""),
        ms4_preset=getattr(S, "ms4_preset", None),
        ks4_preset=getattr(S, "ks4_preset", None),
        sc2_preset=getattr(S, "sc2_preset", None),

        # MS4 params (harmless if other sorter)
        detect_threshold=S.detect_threshold,
        detect_sign=S.detect_sign,
        adjacency_radius=S.adjacency_radius,
        clip_size=S.clip_size,
        detect_interval=getattr(S, "detect_interval", None),   # <-- ADD THIS
        n_jobs=S.n_jobs,
    
        # Preproc
        hp_min_hz=S.hp_min_hz,
        hp_max_hz=S.hp_max_hz,
        use_cmr=S.use_cmr,
    
        # Notch (new canonical + legacy for compatibility)
        use_notch_60hz=getattr(S, "use_notch_60hz", True),
        hp_notch_hz=getattr(S, "hp_notch_hz", 60.0),
        apply_notch=getattr(S, "apply_notch", getattr(S, "use_notch_60hz", True)),
        notch_freq_hz=getattr(S, "notch_freq_hz", getattr(S, "hp_notch_hz", 60.0)),
    )
    out = str(Path(_get_active_run_dir(S.outdir))/"analysis_settings.json")
    import json
    with open(out, "w") as f:
        json.dump(d, f, indent=2)
    _log(f"Saved settings: {out}")

save_meta_btn.on_click(on_save_meta)

# ============================== Layout / Tabs =======================================
hdr   = w.HTML("<h3>MEA/Neuralynx Spike Sorting — Jupyter UI</h3>")
row0  = w.HBox([core_path, core_load_btn, core_status])
geom_row = w.Box([(geom_fc if HAS_FC else geom_tb)])
input_row = w.Box([(h5_fc if HAS_FC else h5_tb)])
_on_input_mode_change()

# Tab 1
t1_inputs = w.VBox([
    w.HTML("<b>Inputs</b>"),
    input_mode_dd,
    input_row,
    tetrode_box,
    geom_row,
    w.HBox([outdir_tb, pick_out, open_out]),
    w.HTML("<hr>"),
    w.HTML("<b>QC (quietest-window RMS)</b>"),
    w.HBox([qc_t0, qc_t1, qc_win]),
    w.HBox([hp_min, hp_max, use_cmr, notch_filter, notch_freq]),
    run_qc,
    w.HTML("<hr>"),
    use_grid,
    qc_tbl_btns,
    qc_table_box,
    w.HBox([qc_flagged_dropdown, qc_plot_btn]),
    qc_plot_out,
    w.HBox([qc_allch_dropdown, qc_plot_notch_btn, qc_plot_cmr_btn]),
    qc_plot_notch_out,
    qc_plot_cmr_out,
])

# Tab 2
t2_sort = w.VBox([
    w.HTML("<b>Sorting</b>"),
    sorter_dd,
    param_stack,          # <-- was ms4_params_box
    run_sort_btn,
    sorter_explain_html,
    w.HTML("<hr>"),
    w.HTML("<b>Compare sorters</b>"),
    w.HBox([cmp_ms4_ks4_btn, cmp_ms4_sc2_btn, cmp_ks4_sc2_btn]),
    cmp_config_panel,
    w.HTML("<hr>"),
    w.HTML("<b>Quality metrics & Reports</b>"),
    w.HBox([compute_metrics_btn, export_spike_times_btn, build_reports_btn]),
    w.HBox([acg_toggle, acg_pdf, pc_metrics]),
])

# Tab 4
t4_curation = w.VBox([
    w.HTML("<b>Two-pass unit curation</b>"),
    w.HTML("<em>Run Tab 2 → Compute metrics first, then either auto-find or select the generated quality_metrics.xlsx.</em>"),
    w.HBox([cur_metrics_path, cur_find_metrics_btn]),
    w.HBox([cur_output_name]),
    w.HTML("<hr>"),
    w.HTML("<b>Curation parameters</b>"),
    w.HBox([cur_amp_max, cur_presence_min, cur_contam_max, cur_nn_min]),
    w.HBox([cur_apply_nn, cur_amp_nan_keep]),
    w.HBox([cur_nn_nan_keep, cur_require_nn]),
    cur_run_btn,
    cur_summary_html,
    w.HTML("<hr>"),
    cur_explain_html,
])

# The old Export/Batch tab is intentionally omitted because it is not used in the current workflow.
tabs = w.Tab(children=[t1_inputs, t2_sort, t4_curation])
tabs.set_title(0, "1) Load & QC")
tabs.set_title(1, "2) Sorting & Metrics")
tabs.set_title(2, "3) Unit Curation")

footer = w.HBox([progress])
display(w.VBox([hdr, row0, tabs, w.HTML("<hr>"), log_out, footer]))